# Phase I (AI group) – *Where does Maia-2 really fail, and what kind of failure is it?*

**Model:** Maia-2 (blitz + rapid) · **Data:** Lichess June 2024, same `moves_maia_2024-06.parquet` as the human-only pair · **Black-box only** (inputs + outputs; the only thing we ever *feed* Maia is positions and Elo labels).

### What this notebook does differently
The human-only pair mapped *aggregate* failure (PCA/ICA, per-square heatmaps, phase × clock). Our two earlier AI notebooks added a failure taxonomy, calibration and value-head checks. This notebook asks the questions that those leave open — **is a "failure" actually an error?** and **what in the situation explains it?**

| # | Question | Core idea |
|---|---|---|
| 1 | Audit | Game-clustered CIs, colour symmetry, Elo-clamp, selection bias, threshold sensitivity |
| 2 | *Is a miss an error?* | Stockfish check of **both** moves → equally good alternative / human error / Maia error |
| 3 | *Noise ceiling* | Repeated positions → how much of the miss rate is irreducible human variability |
| 4 | *Clock- and history-blindness* | Maia sees one board + two Elo labels. Human think-time, recaptures, post-blunder "tilt" |
| 5 | *Does the Elo input do what it should?* | Counterfactual Elo experiment (re-run Maia with swapped ratings) |
| 6 | *Style bias* | Move-type mix of Maia vs humans over **all** positions (not only confident misses) |
| 7 | *Failure atlas + slice dossier* | Interpretable slices; for every slice: what distinguishes it, what Maia does inside it, why it has this much data, whether sub-slices behave differently (confirmed on held-out games) |
| 7b | *Failure modes* | Cluster the misses by their behavioural signature (confidence, rank of human move, think-time…): density, profile, sub-clusters, board examples |
| 7c | *Where in input space?* | Map of ALL positions (board + Elo only): failure vs. rarity/density, regions, and the link region ⇄ failure mode |
| 8 | Dashboard | Single interactive HTML: atlas ⇄ board examples ⇄ every figure above ⇄ AI log |

### How generative AI is used (documented)
Every AI interaction is appended to `ai_phase1_out/ai_log.csv` and rendered in the dashboard's *AI log* tab. The code in this notebook was mainly **written by Claude but has not been run on the real data by Claude**, human oversight, verification and bugfixes followed. The `human_decision` / `verification` columns of the csv explain the human input.

### Requirements
`pip install pandas pyarrow numpy plotly scikit-learn python-chess tqdm maia2 torch` optional: a `stockfish` binary on `PATH`

## 0.1 Setup

In [1]:
import os, re, json, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import chess, chess.svg
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)
pio.templates.default = "plotly_white"
pio.renderers.default = "notebook" 

MONTH = "2024-06"
DATA_DIR = Path("data")
OUT = Path("ai_phase1_out"); OUT.mkdir(exist_ok=True)
SEED = 42
rng = np.random.default_rng(SEED)

C_MAIA, C_HUMAN, C_GREY = "#2a78d6", "#d9534f", "#8a8f98"

# every figure registers itself here; section 8 turns the registry into the dashboard
REG = {}
def register(fig, tab, title, caption="", show=True):
    REG[title] = dict(tab=tab, title=title, caption=caption, fig=fig)
    if show:
        fig.show()

## 0.2 Generative-AI usage log
`log_ai()` appends one row per AI interaction. The first rows document how *this* notebook came about.

In [2]:
LOG_PATH = OUT / "ai_log.csv"
LOG_COLS = ["id", "date", "step", "tool", "prompt_summary", "ai_output_summary", "human_decision", "verification"]

def log_ai(step, tool, prompt_summary, ai_output_summary, human_decision="TO FILL (accepted / modified / rejected + why)",
           verification="TO FILL", date=None):
    n = len(pd.read_csv(LOG_PATH)) + 1 if LOG_PATH.exists() else 1
    row = dict(id=n, date=date or pd.Timestamp.now().strftime("%Y-%m-%d"), step=step, tool=tool,
               prompt_summary=prompt_summary, ai_output_summary=ai_output_summary,
               human_decision=human_decision, verification=verification)
    pd.DataFrame([row], columns=LOG_COLS).to_csv(LOG_PATH, mode="a", header=not LOG_PATH.exists(), index=False)

if not LOG_PATH.exists():     # pre-filled entries describing how this notebook was produced
    T = "Claude (claude.ai chat)"
    log_ai("Planning", T,
           "Task text + two notebooks (human-only pipeline, AI-pair 'pavao'). 'Plan out what I should do.'",
           "Audit + plan. Flags: notebooks use Maia-2 not Maia-1; knight-bias compared on confident misses only; confidence-threshold sweep left undone; "
           "Lichess-analysed games = selected subset; Maia-2 Elo categories clamp <1100 and >=2000; 670k positions but ~10k games (need game-clustered CIs). "
           "Proposed 7 hypotheses and an AI-usage log.", date="2026-11-09")
    log_ai("Review of colleague notebook", T,
           "Reviewed colleague.ipynb (calibration, move quality, value head).",
           "Issues: no CIs; ECE sensitive to confidence distribution; blunder P(true) lacks a matched control; value head predicts human *game outcome* "
           "while engine win-chance is an evaluation -> not the same quantity, 'gap' partly expected; Elo groups differ from Maia's own categories.",
           date="2026-11-09")
    log_ai("Environment check", T, "Asked to 'get to work'.",
           "Tried to fetch data/weights: Lichess dump, Maia-2 weights (Google Drive) and the parquet are unreachable in the AI sandbox -> AI cannot produce results; "
           "decision: write the full analysis as code for the team to run.",
           "n/a – statement of fact", "n/a", date="2026-11-09")
    log_ai("Self-assessment vs. task wording", T,
           "Asked: does the notebook satisfy 'understand patterns in depth: what distinguishes a cluster, why dense/sparse, sub-clusters?'",
           "No, only partly: atlas identified slices but did not explain them; findings table empty; effects not adjusted for confounds; no failure-mode analysis. "
           "Added: slice dossier, adjusted-vs-raw (partial dependence), failure-mode clustering with density + sub-cluster analysis, understanding protocol in section 9.",
           date="2026-11-09")
    log_ai("Notebook generation", T,
           "'Construct a new notebook exploring the tasks you set out; no synthetic data checks, just write code.'",
           "This notebook (sections 1-8). Static checks only (py_compile + pyflakes). No execution on real data.",
           verification="Static checks by AI. Everything else TO FILL – see checklist §0.3.", date="2026-11-09")
if "Input-space map" not in pd.read_csv(LOG_PATH)["step"].values:
    log_ai("Input-space map", "Claude (claude.ai chat)",
           "Self-assessment follow-up: the failure-mode map contained only misses in output space; no input-space map, no density-vs-failure test, no link between where and how.",
           "Added section 7.3: input-space embedding of ALL positions (board + Elo only), kNN sparsity / repeats vs failure, regions with dossiers, region x failure-mode link, input profiles of modes, dashboard tab 'Input space'. Not executed on real data.",
           verification="Static checks (ast/pyflakes) by AI. Everything else TO FILL - see checklist 0.3 item 6.", date="2026-11-09")
pd.read_csv(LOG_PATH)


,id,date,step,tool,prompt_summary,ai_output_summary,human_decision,verification
0,1,2026-11-09,Planning,Claude (claude.ai chat),Task text + two notebooks (human-only pipeline...,Audit + plan. Flags: notebooks use Maia-2 not ...,Accepted,TO FILL
1,2,2026-11-09,Review of colleague notebook,Claude (claude.ai chat),"Reviewed colleague.ipynb (calibration, move qu...",Issues: no CIs; ECE sensitive to confidence di...,"Accepted, then modified with the following prompt",TO FILL
2,3,2026-11-09,Environment check,Claude (claude.ai chat),Asked to 'get to work'.,"Tried to fetch data/weights: Lichess dump, Mai...",n/a – statement of fact,NaN
3,4,2026-11-09,Self-assessment vs. task wording,Claude (claude.ai chat),Asked: does the notebook satisfy 'understand p...,"No, only partly: atlas identified slices but d...",Accepted,TO FILL
4,5,2026-11-09,Notebook generation,Claude (claude.ai chat),'Construct a new notebook exploring the tasks ...,This notebook (sections 1-8). Static checks on...,"Accepted, later modified",Static checks by AI. Everything else TO FILL –...
5,6,2026-11-09,Input-space map,Claude (claude.ai chat),Self-assessment follow-up: the failure-mode ma...,Added section 7.3: input-space embedding of AL...,Accepted,Static checks (ast/pyflakes) by AI. Everything...


## 0.3 Verification checklist for the human (do this – AI-written code can be confidently wrong)
1. §1: the printed row counts, top-1 accuracy (0.50) and top-3 (0.78) must match your earlier notebooks.
2. §1: `cluster_rates` – compare one rate to a plain `df.groupby().mean()`; the CI should be **wider** than a naive binomial CI.
3. §3: hand-check the leave-one-out consensus on one `key` (use the cell provided).
4. §5: with *true* Elo, the log-likelihood from the experiment should match `maia_p_true` from the parquet (cell provided).
5. §2: eyeball 10 Stockfish-labelled disagreements on a board before trusting the category shares.
6. §7.3: check that `IS['failed'].mean()` ≈ the global failure rate, that the embedding is not one blob (try `EMB_IN`), and look at 5 boards of one region in the dashboard to see whether the region label matches what you see.
7. Log every correction you make in `ai_log.csv` – AI mistakes are findings of this assignment, too.

## 0.4 Hypotheses (written **before** looking at results)
| | Hypothesis | If true we expect… | …and we reject it if |
|---|---|---|---|
| H1 | A large share of top-1 "failures" are *equally good alternatives* | Many misses where both Maia's move and the human's move lose < tol vs. Stockfish's best | Misses are mostly "human error" or "Maia error" |
| H2 | A sizeable part of the miss rate is irreducible human variability | In repeated positions a leave-one-out *consensus* predictor is only a little better than Maia | Consensus predictor ≫ Maia (model error) |
| H3 | Maia is clock-blind: failures concentrate at extreme think-times (premoves / long thinks) and Maia's uncertainty does not track human deliberation | Failure U-shaped in think-time; weak entropy↔think-time rank-correlation | Flat failure vs. think-time |
| H4 | Maia is history-blind: recaptures and moves after a blunder behave differently from what the board alone predicts | Failure differs for `is_recapture`, after own/opponent blunder, controlling for position | No difference |
| H5 | The Elo inputs work, but weakly at the clamped edges and for the *opponent* rating | Log-likelihood peaks at true self-Elo; flat/no peak at `<1100` and `>=2000`; opponent Elo has small effect | Peak is not on the diagonal |
| H6 | Maia has a systematic style bias (move-type mix) that is visible over **all** positions | Maia/human rate gap for knight, capture, castle, forward/backward moves ≠ 0 with CI excluding 0 | No gaps beyond CI |
| H7 | Failures are concentrated in identifiable slices (sharp positions, many legal moves, unbalanced material) | Tree/slice lift ≫ 1; a model using only *conditions* has AUC clearly > 0.5 | AUC ≈ 0.5 |
| H9 | Misses are not one phenomenon: they form a few qualitatively different *failure modes* | Clusters with clearly different profiles, densities and context lifts; some with real sub-structure (silhouette) | One diffuse cloud (low silhouette), same context everywhere |
| H10 | Failure depends on *where* the position lies in input space, and the failure modes concentrate in different regions | Failure higher in rare/sparse regions (CI excludes the dense rate); Cramér's V(region, mode) clearly > 0; input-only AUC > 0.5 | Failure flat across density/popularity and V ≈ 0 |
| H8 | Results are not artefacts of Elo clamping or of the "analysed games" subset | Same patterns with/without clamped bins, with/without eval | Patterns change |

## 1. Load data, engineer features, build helpers
Beyond the existing columns we derive: **history features** (recapture, own/opponent previous move quality), **position features** (material balance, piece count, queens on, in-check), and **move descriptors for both the human move and Maia's move** (piece, capture, castle, promotion, direction, check, distance). The python-chess loop takes a minute or two and is cached.

In [3]:
CANDIDATES = [DATA_DIR / f"moves_maia_{MONTH}.parquet", Path("../non-ai-phase1/data") / f"moves_maia_{MONTH}.parquet"]
DATA = next((p for p in CANDIDATES if p.exists()), None)
assert DATA is not None, "moves_maia parquet not found"
moves = pd.read_parquet(DATA).sort_values(["game_id", "ply"]).reset_index(drop=True)
moves["failed"] = ~moves["maia_correct"].astype(bool)

def elo_sort_key(s):
    nums = re.findall(r"\d+", str(s))
    return -1 if not nums else int(nums[0]) - (0.5 if str(s).startswith("<") else 0)
ELO_ORDER = sorted(moves["maia_elo_cat_self"].astype(str).unique(), key=elo_sort_key)
CLAMPED = [e for e in ELO_ORDER if e.startswith("<") or e.startswith(">")]
PHASES = ["opening", "middlegame", "endgame"]
QUAL = ["good", "inaccuracy", "mistake", "blunder"]

# --- history (needs the ply order inside a game) ---
g = moves.groupby("game_id", sort=False)
moves["prev_move"] = g["move"].shift(1)
prev_cap = g["is_capture"].shift(1).fillna(False).astype(bool)
moves["is_recapture"] = moves["is_capture"].astype(bool) & prev_cap & (moves["move"].str[2:4] == moves["prev_move"].str[2:4])
moves["opp_prev_quality"] = g["move_quality"].shift(1)
moves["own_prev_quality"] = g["move_quality"].shift(2)

# --- bins used all over the place ---
moves["think_bin"] = pd.cut(moves["time_spent"], [-0.01, 1, 3, 10, 30, np.inf], labels=["≤1s", "1–3s", "3–10s", "10–30s", ">30s"])
moves["clock_group"] = pd.cut(moves["clock_before"], [-np.inf, 30, 60, 180, np.inf], labels=["<30s", "30–60s", "1–3 min", ">3 min"])
moves["state"] = pd.cut(moves["wc_before"], [-1.01, -0.5, -0.15, 0.15, 0.5, 1.01], labels=["lost", "worse", "equal", "better", "winning"])
moves["elo_diff_bin"] = pd.cut(moves["elo_diff"], [-np.inf, -200, -50, 50, 200, np.inf], labels=["≪ lower", "lower", "even", "higher", "≫ higher"])
moves["legal_bin"] = pd.cut(moves["n_legal"], [0, 15, 25, 35, np.inf], labels=["≤15", "16–25", "26–35", ">35"])
moves["move_bucket"] = pd.cut(moves["move_number"], [0, 5, 12, 20, 30, 45, np.inf], labels=["1–5", "6–12", "13–20", "21–30", "31–45", "46+"])

# --- position features straight from the FEN ---
board_part = moves["fen"].str.split(" ").str[0]
vals = {"p": 1, "n": 3, "b": 3, "r": 5, "q": 9}
w_mat = sum(board_part.str.count(k.upper()) * v for k, v in vals.items())
b_mat = sum(board_part.str.count(k) * v for k, v in vals.items())
sgn = np.where(moves["side"] == "white", 1, -1)
moves["mat_balance"] = sgn * (w_mat - b_mat)
moves["n_pieces"] = board_part.str.count(r"[a-zA-Z]")
moves["queens_on"] = board_part.str.contains("[Qq]")

print(f"{len(moves):,} positions | {moves['game_id'].nunique():,} games | Elo categories: {ELO_ORDER}")

676,579 positions | 10,000 games | Elo categories: ['<1100', '1100-1199', '1200-1299', '1300-1399', '1400-1499', '1500-1599', '1600-1699', '1700-1799', '1800-1899', '1900-1999', '>=2000']


In [4]:
# --- move descriptors for BOTH the human move and Maia's top-1 move (python-chess loop, cached) ---
MF_NAMES = ["piece", "capture", "castle", "promo", "dir", "check", "dist"]
MF_PATH = OUT / f"move_features_{MONTH}.parquet"

def describe(board, uci):
    m = chess.Move.from_uci(uci)
    p = board.piece_at(m.from_square)
    if p is None:
        return (None, False, False, False, 0, False, 0)
    dr = (chess.square_rank(m.to_square) - chess.square_rank(m.from_square)) * (1 if board.turn else -1)
    return (p.symbol().upper(), board.is_capture(m), board.is_castling(m), m.promotion is not None,
            int(np.sign(dr)), board.gives_check(m), chess.square_distance(m.from_square, m.to_square))

MF_COLS = ["in_check"] + [f"h_{c}" for c in MF_NAMES] + [f"m_{c}" for c in MF_NAMES]
if MF_PATH.exists() and len(pd.read_parquet(MF_PATH)) == len(moves):
    mf = pd.read_parquet(MF_PATH)
else:
    rows = []
    for fen, hm, mm in tqdm(zip(moves["fen"], moves["move"], moves["maia_move"]), total=len(moves)):
        b = chess.Board(fen)
        rows.append((b.is_check(),) + describe(b, hm) + describe(b, mm))
    mf = pd.DataFrame(rows, columns=MF_COLS)
    mf.to_parquet(MF_PATH, index=False)
for c in MF_COLS:
    moves[c] = mf[c].values

def mclass(piece, cap, castle, promo):
    if castle: return "castle"
    if promo: return "promotion"
    return f"{piece}{'x' if cap else ''}"
moves["h_class"] = [mclass(*t) for t in zip(moves["h_piece"], moves["h_capture"], moves["h_castle"], moves["h_promo"])]
moves["m_class"] = [mclass(*t) for t in zip(moves["m_piece"], moves["m_capture"], moves["m_castle"], moves["m_promo"])]

d = moves[~moves["forced"]].copy()            # analysis frame: non-forced positions
print(f"non-forced: {len(d):,} | top-1 {d['maia_correct'].mean():.3f} | top-3 {d['maia_top3_hit'].mean():.3f} | top-5 {d['maia_top5_hit'].mean():.3f}")

non-forced: 669,759 | top-1 0.498 | top-3 0.780 | top-5 0.878


In [5]:
# ---------- helpers ----------
def cluster_rates(df, by, val="failed", B=300, seed=0):
    # mean of `val` per group in `by`, with a 95% CI from a bootstrap that resamples whole GAMES
    # (positions inside a game are strongly correlated, so a plain binomial CI is far too narrow)
    by = [by] if isinstance(by, str) else list(by)
    df = df.dropna(subset=by + [val])
    labels = df[by[0]].astype(str) if len(by) == 1 else df[by].astype(str).agg("|".join, axis=1)
    gcode, gids = pd.factorize(df["game_id"]); kcode, kids = pd.factorize(labels)
    G, K = len(gids), len(kids)
    num = np.zeros((G, K)); den = np.zeros((G, K))
    np.add.at(num, (gcode, kcode), df[val].to_numpy(float)); np.add.at(den, (gcode, kcode), 1.0)
    W = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=B).astype(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        R = (W @ num) / (W @ den)
        rate = num.sum(0) / den.sum(0)
    out = pd.DataFrame({"n": den.sum(0).astype(int), "rate": rate,
                        "lo": np.nanpercentile(R, 2.5, axis=0), "hi": np.nanpercentile(R, 97.5, axis=0)}, index=kids)
    if len(by) > 1:
        parts = pd.DataFrame([k.split("|") for k in out.index], columns=by, index=out.index)
        out = pd.concat([parts, out], axis=1)
    return out

def ci_idx(gcode_all, y, idx, B=300, seed=0):
    # CI for the mean of y over rows `idx` (positional), bootstrap over games
    G = gcode_all.max() + 1
    num = np.bincount(gcode_all[idx], weights=y[idx], minlength=G); den = np.bincount(gcode_all[idx], minlength=G).astype(float)
    W = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=B).astype(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        R = (W @ num) / (W @ den)
    return np.nanpercentile(R, 2.5), np.nanpercentile(R, 97.5)

def heat_trace(r, rows, cols, row_order, col_order, min_n=300, name="failure rate", zmin=None, zmax=None, zmid=None, colorscale="RdBu_r"):
    p = lambda v: r.pivot(index=rows, columns=cols, values=v).reindex(index=row_order, columns=col_order)
    rate, n, lo, hi = p("rate"), p("n"), p("lo"), p("hi")
    z = rate.mask(n < min_n).values.astype(float)
    text = np.where(np.isnan(z), "", np.char.mod("%.2f", z))
    return go.Heatmap(z=z, x=col_order, y=row_order, text=text, texttemplate="%{text}",
                      customdata=np.dstack([n.values, lo.values, hi.values]),
                      hovertemplate=rows + "=%{y}<br>" + cols + "=%{x}<br>" + name + "=%{z:.3f} [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<br>n=%{customdata[0]:,.0f}<extra></extra>",
                      colorscale=colorscale, zmin=zmin, zmax=zmax, zmid=zmid, colorbar=dict(title=name))

def wc(cp):
    cp = np.clip(cp, -1000, 1000)
    return 2 / (1 + np.exp(-0.00368208 * cp)) - 1        # same win-chance scale as the pipeline ([-1, 1])

GLOBAL_FAIL = d["failed"].mean()
d["game_code"] = pd.factorize(d["game_id"])[0]

In [6]:
# sanity check 1 (do it!): cluster_rates must give the same point estimate as groupby, but a WIDER CI than a naive binomial one
chk = cluster_rates(d, "phase").loc[PHASES]
naive = d.groupby("phase")["failed"].agg(["mean", "size"]).loc[PHASES]
chk["naive_halfwidth"] = 1.96 * np.sqrt(naive["mean"] * (1 - naive["mean"]) / naive["size"])
chk["cluster_halfwidth"] = (chk["hi"] - chk["lo"]) / 2
display(chk.round(4)); assert np.allclose(chk["rate"], naive["mean"])

,n,rate,lo,hi,naive_halfwidth,cluster_halfwidth
opening,239456,0.5548,0.5527,0.5570,0.0020,0.0021
middlegame,279110,0.4873,0.4853,0.4895,0.0019,0.0021
endgame,151193,0.4470,0.4438,0.4506,0.0025,0.0034


## 1.1 Audit of the inherited analysis (AI-assisted)
Each check below is a place where an earlier notebook's conclusion could be an artefact. **Read the output before using any of the later results.**

In [7]:
# (a) colour symmetry – Maia mirrors the board for Black; a bug there would look like a "model behaviour"
display(cluster_rates(d, ["side", "phase"]).round(4))

# (b) the 'confident miss' threshold: nothing special happens at 0.6 – show the whole sweep
rows = []
q = d.dropna(subset=["move_quality"])
for thr in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    cm = (q["maia_p_top1"] >= thr) & q["failed"]
    rows.append(dict(threshold=thr, share_of_positions=(d["maia_p_top1"] >= thr).mean(),
                     miss_rate_when_confident=d.loc[d["maia_p_top1"] >= thr, "failed"].mean(),
                     blunder_rate_in_confident_misses=(q.loc[cm, "move_quality"] == "blunder").mean(),
                     blunder_rate_elsewhere=(q.loc[~cm, "move_quality"] == "blunder").mean()))
display(pd.DataFrame(rows).round(3))

# (c) selection: do positions with / without a Lichess engine eval differ?
d["has_eval"] = d["wc_before"].notna()
display(cluster_rates(d, ["has_eval", "phase"]).round(4))

,side,phase,n,rate,lo,hi
white|opening,white,opening,119800,0.5625,0.5600,0.5650
black|opening,black,opening,119656,0.5470,0.5441,0.5498
white|middlegame,white,middlegame,142231,0.4889,0.4861,0.4913
black|middlegame,black,middlegame,136879,0.4856,0.4827,0.4886
black|endgame,black,endgame,75596,0.4472,0.4427,0.4511
white|endgame,white,endgame,75597,0.4468,0.4419,0.4516


,threshold,share_of_positions,miss_rate_when_confident,blunder_rate_in_confident_misses,blunder_rate_elsewhere
0,0.4,0.668,0.394,0.109,0.044
1,0.5,0.504,0.328,0.119,0.049
2,0.6,0.363,0.256,0.132,0.053
3,0.7,0.258,0.195,0.149,0.056
4,0.8,0.172,0.139,0.168,0.058
5,0.9,0.097,0.082,0.187,0.059


,has_eval,phase,n,rate,lo,hi
False|opening,False,opening,10000,0.9646,0.9608,0.9681
True|opening,True,opening,229456,0.5369,0.5348,0.5392
True|middlegame,True,middlegame,279110,0.4873,0.4853,0.4895
True|endgame,True,endgame,151193,0.4470,0.4438,0.4506


In [8]:
# (d) Elo clamping – Maia-2 only knows categories <1100 … >=2000: players outside are out-of-distribution
r = cluster_rates(d, "maia_elo_cat_self").reindex(ELO_ORDER)
fig = go.Figure(go.Bar(x=r.index, y=r["rate"], marker_color=[C_GREY if e in CLAMPED else C_MAIA for e in r.index],
                       error_y=dict(type="data", symmetric=False, array=r["hi"] - r["rate"], arrayminus=r["rate"] - r["lo"]),
                       customdata=r["n"], hovertemplate="%{x}<br>failure %{y:.3f}<br>n=%{customdata:,}<extra></extra>"))
fig.add_hline(y=GLOBAL_FAIL, line_dash="dash", line_color="black")
fig.update_layout(title="Top-1 failure rate by Maia's own Elo category (95% CI, bootstrap over games; grey = clamped categories)",
                  yaxis_title="failure rate", xaxis_title="player Elo category", height=420)
register(fig, "Audit", "Failure by Elo category", "Grey bars are the two clamped end categories (everyone <1100 / ≥2000 shares one input).")

## 2. Is a "miss" an error? – anatomy of a failure (needs Stockfish)
Top-1 accuracy treats *any* difference between Maia's move and the human's move as a failure. But in many positions several moves are about equally good. For a stratified sample we ask Stockfish about **three** things: the best move/second-best move (position *sharpness*), the value of the **human** move and the value of **Maia's** move. Each miss then falls into one of

* **equally good alternative** – both moves lose ≤ `TOL` against the best move;
* **human error (Maia sound)** – the human move is worse by > `TOL`, Maia's is fine → Maia is *right*, the top-1 metric punishes it;
* **Maia error (human sound)** – Maia's move is worse by > `TOL`, the human's is fine → a *real* model failure;
* **both inaccurate**.

Install: `apt install stockfish` / `brew install stockfish`. If no binary is found, this section is skipped and the dashboard just omits it.

In [9]:
import concurrent.futures as cf
SF_PATH = shutil.which("stockfish")          # or set manually: SF_PATH = "/path/to/stockfish"
N_SF, SF_DEPTH, SF_CHUNK = 24000, 12, 200
N_PROC = max(1, (os.cpu_count() or 2) - 1)
SF_CACHE = OUT / f"stockfish_{MONTH}_n{N_SF}_d{SF_DEPTH}.parquet"

SF_WORKER_SRC = """
import chess, chess.engine
def analyse_chunk(args):
    path, depth, chunk = args
    eng = chess.engine.SimpleEngine.popen_uci(path)
    try:
        eng.configure({"Threads": 1, "Hash": 64})
    except Exception:
        pass
    lim = chess.engine.Limit(depth=depth)
    out = []
    for idx, fen, hm, mm in chunk:
        b = chess.Board(fen)
        try:
            info = eng.analyse(b, lim, multipv=2)
            sc = [i["score"].pov(b.turn).score(mate_score=10000) for i in info]
            roots = [chess.Move.from_uci(m) for m in sorted({hm, mm})]
            info2 = eng.analyse(b, lim, root_moves=roots, multipv=len(roots))
            vals = {i["pv"][0].uci(): i["score"].pov(b.turn).score(mate_score=10000) for i in info2}
            out.append((idx, sc[0], sc[1] if len(sc) > 1 else None, vals.get(hm), vals.get(mm)))
        except Exception:
            out.append((idx, None, None, None, None))
    eng.quit()
    return out
"""
Path("sf_worker.py").write_text(SF_WORKER_SRC)      # separate module so it can be pickled by worker processes

sf = None
if SF_CACHE.exists():
    sf = pd.read_parquet(SF_CACHE)
elif SF_PATH:
    import importlib, sf_worker; importlib.reload(sf_worker)
    per = N_SF // 3
    smp = d.groupby("phase", group_keys=False).apply(lambda x: x.sample(min(len(x), per), random_state=SEED))
    tasks = [(SF_PATH, SF_DEPTH, list(zip(c.index, c["fen"], c["move"], c["maia_move"])))
             for c in (smp.iloc[i:i + SF_CHUNK] for i in range(0, len(smp), SF_CHUNK))]
    res = []
    with cf.ProcessPoolExecutor(N_PROC) as ex:
        for fut in tqdm(cf.as_completed([ex.submit(sf_worker.analyse_chunk, t) for t in tasks]), total=len(tasks)):
            res += fut.result()
    sf = pd.DataFrame(res, columns=["idx", "sf_best", "sf_second", "sf_h", "sf_m"]).set_index("idx")
    sf.to_parquet(SF_CACHE)
else:
    print("No Stockfish found -> section 2 skipped (set SF_PATH to enable).")
if sf is not None:
    print(f"{len(sf):,} positions analysed, {sf['sf_h'].isna().sum()} failed")

24,000 positions analysed, 0 failed


In [10]:
CATS = ["agree", "equally good alternative", "human error (Maia sound)", "Maia error (human sound)", "both inaccurate"]
CAT_COL = dict(zip(CATS, ["#b8c4d6", "#7fb77e", "#f2c14e", C_HUMAN, "#7a4b94"]))
TOL = 0.05                                       # win-chance units on [-1, 1]; 0.05 ~ 2.5 percentage points of win probability

def classify(a, tol):
    hf, mf_ = a["h_loss"] <= tol, a["m_loss"] <= tol
    return np.select([~a["failed"], hf & mf_, ~hf & mf_, hf & ~mf_], CATS[:4], CATS[4])

a = None
if sf is not None:
    a = d.join(sf, how="inner").dropna(subset=["sf_best", "sf_h", "sf_m"]).copy()
    best = np.maximum.reduce([a["sf_best"], a["sf_h"], a["sf_m"]])        # guard against search noise between the two analyses
    a["h_loss"] = wc(best) - wc(a["sf_h"]); a["m_loss"] = wc(best) - wc(a["sf_m"])
    a["gap"] = wc(best) - wc(a["sf_second"].fillna(-1000))                # how much worse the 2nd-best move is: big = sharp position
    a["sharp_bin"] = pd.qcut(a["gap"], 4, labels=["Q1 flat (many good moves)", "Q2", "Q3", "Q4 sharp (few good moves)"])
    a["fail_cat"] = classify(a, TOL)
    a["maia_error"] = a["fail_cat"].eq(CATS[3]).astype(int)
    fails = a[a["failed"]]
    head = fails["fail_cat"].value_counts(normalize=True).reindex(CATS[1:]).round(3)
    print(f"{len(a):,} analysed positions, miss rate {a['failed'].mean():.3f}. Share of MISSES by category (TOL={TOL}):")
    display(head.to_frame("share_of_misses"))
    sens = pd.DataFrame({tol: pd.Series(classify(a, tol)[a["failed"].values]).value_counts(normalize=True).reindex(CATS[1:]) for tol in (0.025, 0.05, 0.1)})
    display(sens.round(3).rename_axis("category").rename(columns=lambda c: f"TOL={c}"))

24,000 analysed positions, miss rate 0.500. Share of MISSES by category (TOL=0.05):


,share_of_misses
fail_cat,
equally good alternative,0.401
human error (Maia sound),0.246
Maia error (human sound),0.169
both inaccurate,0.184


,TOL=0.025,TOL=0.05,TOL=0.1
category,,,
equally good alternative,0.241,0.401,0.598
human error (Maia sound),0.268,0.246,0.192
Maia error (human sound),0.184,0.169,0.117
both inaccurate,0.307,0.184,0.092


In [11]:
print(type(a), None if a is None else a.shape)
print([t for t in REG if "Failure categories" in t or "Human loss" in t or "Sharpness" in t])
import plotly.io as pio; print(pio.renderers.default)
go.Figure(go.Bar(y=[1, 3, 2])).show()

<class 'pandas.core.frame.DataFrame'> (24000, 94)
[]
notebook


In [12]:
if a is not None:
    # --- Figure: stacked category shares, switchable by condition ---
    VARS = [("phase", PHASES), ("maia_elo_cat_self", ELO_ORDER), ("sharp_bin", [str(c) for c in a["sharp_bin"].cat.categories]),
            ("think_bin", [str(c) for c in a["think_bin"].cat.categories]), ("legal_bin", [str(c) for c in a["legal_bin"].cat.categories]),
            ("state", [str(c) for c in a["state"].cat.categories])]
    fig = go.Figure()
    for vi, (v, order) in enumerate(VARS):
        ct = pd.crosstab(a[v].astype(str), a["fail_cat"], normalize="index").reindex(index=order, columns=CATS).fillna(0)
        for c in CATS:
            fig.add_trace(go.Bar(x=ct.index, y=ct[c], name=c, marker_color=CAT_COL[c], visible=(vi == 0), showlegend=(vi == 0)))
    nC = len(CATS)
    fig.update_layout(barmode="stack", height=470, yaxis_title="share of all positions", title="What a top-1 'miss' really is (Stockfish check of both moves)",
        updatemenus=[dict(type="dropdown", x=0, y=1.15, xanchor="left", buttons=[
            dict(label=v, method="update", args=[{"visible": [j // nC == vi for j in range(len(VARS) * nC)],
                                                  "showlegend": [j // nC == vi for j in range(len(VARS) * nC)]}]) for vi, (v, _) in enumerate(VARS)])])
    register(fig, "Anatomy of a failure", "Failure categories by condition",
             "Use the dropdown to change the conditioning variable. Green+yellow are misses that are *not* model errors; red is a real Maia error.")

    # --- Figure: human loss vs Maia loss on misses ---
    f = a[a["failed"]]
    fig = go.Figure(go.Histogram2d(x=f["h_loss"].clip(0, 1), y=f["m_loss"].clip(0, 1), nbinsx=40, nbinsy=40, colorscale="Blues",
                                   colorbar=dict(title="misses")))
    for v in (TOL,):
        fig.add_vline(x=v, line_dash="dash"); fig.add_hline(y=v, line_dash="dash")
    fig.update_layout(title="Misses only: how much worse than Stockfish's best is each move?", xaxis_title="human move: win-chance lost",
                      yaxis_title="Maia move: win-chance lost", height=470)
    register(fig, "Anatomy of a failure", "Human loss vs Maia loss",
             "Bottom-left quadrant = equally good alternatives; bottom-right = human erred; top-left = Maia erred; top-right = both.")

    # --- Figure: sharpness vs failure (does the engine-defined sharpness explain misses?) ---
    rows = []
    for lab, sub in a.groupby("sharp_bin"):
        rows.append(dict(sharp=str(lab), any_miss=sub["failed"].mean(), maia_error=(sub["fail_cat"] == CATS[3]).mean(),
                         human_error=(sub["fail_cat"] == CATS[2]).mean(), alt=(sub["fail_cat"] == CATS[1]).mean()))
    s = pd.DataFrame(rows)
    fig = go.Figure([go.Scatter(x=s["sharp"], y=s[c], mode="lines+markers", name=n, line=dict(color=col)) for c, n, col in
                     [("any_miss", "any top-1 miss", "black"), ("maia_error", "Maia error", C_HUMAN), ("human_error", "human error", "#f2c14e"), ("alt", "equally good alternative", "#7fb77e")]])
    fig.update_layout(title="Position sharpness (gap between best and 2nd-best move) vs. failure type", yaxis_title="share of positions", height=420)
    register(fig, "Anatomy of a failure", "Sharpness vs failure type", "In flat positions many moves are fine so misses are cheap; do real Maia errors rise in sharp positions?")

## 3. Noise ceiling – how much of the miss rate is irreducible?
Positions that occur in many games (mostly openings) let us estimate what *the human population* does. For each such position we build a **leave-one-out consensus** (the most common move among the *other* games) and compare three things per row: Maia's move, the consensus move, the move actually played. This splits misses into

* **individual deviation** – Maia agrees with the consensus; this player simply did something else (no model can win these),
* **Maia off-consensus** – the player followed the consensus, Maia did not (model error relative to the population),
* **nobody agrees** – long-tail human choice.

*Limitation:* only positions seen ≥ `N_MIN` times in different games qualify, so this section speaks mainly about the first ~10-15 moves.

In [13]:
N_MIN = 10
d["key"] = d["fen"].str.split(" ").str[:4].str.join(" ")           # board, side, castling, en-passant (move counters dropped)
nk = d.groupby("key")["game_id"].nunique()
k = d[d["key"].isin(nk[nk >= N_MIN].index)].copy()
print(f"{len(k):,} of {len(d):,} positions ({len(k)/len(d):.1%}) in {k['key'].nunique():,} positions seen in >= {N_MIN} games")
display(k.groupby("phase").size().div(d.groupby("phase").size()).rename("coverage").round(3).to_frame())

idxs, loo, share, ent, kn_ = [], [], [], [], []
for key, sub in k.groupby("key"):
    vc = sub["move"].value_counts(); n = len(sub); top = vc.index[0]; c1 = vc.iloc[0]
    second, c2 = (vc.index[1], vc.iloc[1]) if len(vc) > 1 else (top, 0)
    own_top = (sub["move"] == top).to_numpy()
    idxs += list(sub.index); kn_ += [n] * n; share += [c1 / n] * n
    loo += list(np.where(own_top & (c2 > c1 - 1), second, top))            # remove the row's own vote
    p = (vc / n).to_numpy(); ent += [float(-(p * np.log(p)).sum())] * n
ex_ = pd.DataFrame({"loo_mode": loo, "top_share": share, "emp_entropy": ent, "key_n": kn_}, index=idxs)
k = k.join(ex_)
k["loo_hit"] = k["loo_mode"] == k["move"]
k["maia_eq_mode"] = k["maia_move"] == k["loo_mode"]
k["fail_kind"] = np.select([~k["failed"], k["failed"] & k["maia_eq_mode"], k["failed"] & ~k["maia_eq_mode"] & k["loo_hit"]],
                           ["correct", "individual deviation (Maia = consensus)", "Maia off-consensus (human = consensus)"],
                           "nobody agrees (long-tail human choice)")
print(f"on these positions: Maia top-1 acc {1 - k['failed'].mean():.3f} | leave-one-out consensus predictor acc {k['loo_hit'].mean():.3f}")

63,496 of 669,759 positions (9.5%) in 1,012 positions seen in >= 10 games


,coverage
phase,
endgame,NaN
middlegame,NaN
opening,0.265


on these positions: Maia top-1 acc 0.288 | leave-one-out consensus predictor acc 0.519


In [14]:
# sanity check 3 (do it!): hand-verify the leave-one-out consensus on one position
demo = k["key"].value_counts().index[0]; sub = k[k["key"] == demo]
print("position:", demo, "\nmove counts:", sub["move"].value_counts().head(4).to_dict())
print("loo_mode per own move:", sub.groupby("move")["loo_mode"].first().head(4).to_dict(), "  <- must equal the top move unless the row's own move is the top move AND it leads by exactly 1")

position: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 
move counts: {'e2e4': 6078, 'd2d4': 2629, 'c2c4': 369, 'g1f3': 354}
loo_mode per own move: {'a2a3': np.str_('e2e4'), 'a2a4': np.str_('e2e4'), 'b1a3': np.str_('e2e4'), 'b1c3': np.str_('e2e4')}   <- must equal the top move unless the row's own move is the top move AND it leads by exactly 1


In [15]:
CONS_BINS = [0, 0.3, 0.5, 0.7, 0.9, 1.01]; CONS_LAB = ["<30% (diffuse)", "30–50%", "50–70%", "70–90%", "≥90% (near-deterministic)"]
k["consensus"] = pd.cut(k["top_share"], CONS_BINS, labels=CONS_LAB, right=False)
KINDS = ["correct", "individual deviation (Maia = consensus)", "Maia off-consensus (human = consensus)", "nobody agrees (long-tail human choice)"]
KCOL = ["#b8c4d6", "#7fb77e", C_HUMAN, "#7a4b94"]

fig = make_subplots(1, 2, shared_yaxes=True, subplot_titles=["by strength of human consensus in that position", "by move number"])
for ci, (var, order) in enumerate([("consensus", CONS_LAB), ("move_bucket", [str(c) for c in k["move_bucket"].cat.categories])], start=1):
    ct = pd.crosstab(k[var].astype(str), k["fail_kind"], normalize="index").reindex(index=order, columns=KINDS).fillna(0)
    for kind, col in zip(KINDS, KCOL):
        fig.add_trace(go.Bar(x=ct.index, y=ct[kind], name=kind, marker_color=col, legendgroup=kind, showlegend=(ci == 1)), 1, ci)
    g_ = k.groupby(k[var].astype(str)).agg(maia=("failed", lambda s: 1 - s.mean()), cons=("loo_hit", "mean")).reindex(order)
    fig.add_trace(go.Scatter(x=g_.index, y=g_["maia"], mode="lines+markers", name="Maia top-1 accuracy", line=dict(color="black"), legendgroup="a", showlegend=(ci == 1)), 1, ci)
    fig.add_trace(go.Scatter(x=g_.index, y=g_["cons"], mode="lines+markers", name="consensus-predictor accuracy", line=dict(color="black", dash="dot"), legendgroup="c", showlegend=(ci == 1)), 1, ci)
fig.update_layout(barmode="stack", height=480, yaxis_title="share of positions", title="Noise ceiling: who disagrees with whom in repeatedly-seen positions")
register(fig, "Noise ceiling", "Consensus decomposition", "If the dotted line (consensus) barely exceeds the solid one (Maia), the remaining misses are mostly irreducible human variability.")

# --- key-level calibration: Maia's confidence vs how often humans actually play that move in that position ---
pred = k.groupby(["key", "maia_move"]).agg(n_pred=("maia_p_top1", "size"), conf=("maia_p_top1", "mean")).reset_index()
cnt = k.groupby(["key", "move"]).size().rename("n_played").reset_index().rename(columns={"move": "maia_move"})
pred = pred.merge(cnt, on=["key", "maia_move"], how="left").fillna({"n_played": 0}).merge(k.groupby("key").size().rename("key_n").reset_index(), on="key")
pred["emp"] = pred["n_played"] / pred["key_n"]
pred = pred[pred["n_pred"] >= 3]
mv_no = k.groupby("key")["move_number"].min().rename("first_move_no"); pred = pred.merge(mv_no, on="key")
fig = go.Figure(go.Scatter(x=pred["conf"], y=pred["emp"], mode="markers", marker=dict(size=np.sqrt(pred["n_pred"]) * 2, opacity=0.5, color=pred["first_move_no"],
                colorscale="Viridis", colorbar=dict(title="move no.")), customdata=np.c_[pred["key_n"], pred["n_pred"], pred["maia_move"]],
                hovertemplate="Maia move %{customdata[2]}<br>Maia conf %{x:.2f} | humans play it %{y:.2f}<br>position seen %{customdata[0]}× (Maia predicted it %{customdata[1]}×)<extra></extra>"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="black"), showlegend=False))
fig.update_layout(title="Per position: Maia's confidence in its move vs. the population frequency of that move", xaxis_title="mean Maia P(top-1)",
                  yaxis_title="share of games in which humans played that move", height=520, showlegend=False)
register(fig, "Noise ceiling", "Position-level calibration", "Points below the diagonal: Maia more confident than the human population is. Colour = move number.")

## 4. Clock- and history-blindness
Maia-2's inputs are **one board and two Elo categories** – no clock, no move history. Anything the player does *because of* the clock or the previous moves is invisible to it. We use the **think-time** of each move as a proxy for "how hard did the human find this?" and the previous moves of the game for history effects:

* does Maia's own uncertainty (`maia_entropy_norm`) track human deliberation?
* are very fast moves (premoves/recaptures) and very long thinks both hard to predict?
* is a **recapture** predicted worse/better, and does a player's behaviour change after **their own** or **the opponent's** blunder?

In [16]:
THINK = [str(c) for c in d["think_bin"].cat.categories]; CLK = [str(c) for c in d["clock_group"].cat.categories]
d["ent_q"] = pd.qcut(d["maia_entropy_norm"].rank(method="first"), 4, labels=["Maia very sure", "sure", "unsure", "very unsure"])
ENT = [str(c) for c in d["ent_q"].cat.categories]

fig = make_subplots(1, 3, subplot_titles=["think-time × phase", "think-time × clock left", "think-time × Maia's own uncertainty"], horizontal_spacing=0.12)
for i, (rows_, order, nm) in enumerate([("phase", PHASES, "phase"), ("clock_group", CLK, "clock_group"), ("ent_q", ENT, "ent_q")], start=1):
    r = cluster_rates(d, [rows_, "think_bin"])
    t = heat_trace(r, rows_, "think_bin", order, THINK, zmin=0.2, zmax=0.8, zmid=GLOBAL_FAIL)
    t.showscale = (i == 3); fig.add_trace(t, 1, i)
fig.update_layout(height=430, title="Top-1 failure rate vs. how long the human thought (rate; cells with n<300 hidden)")
register(fig, "Clock & history", "Failure vs think-time", "Right panel: where Maia is *very sure* but the human thought for a long time, something is going on that the board alone does not reveal.")

# does Maia's uncertainty track human deliberation?  Spearman within phase × Elo, then pooled
sm = d.sample(min(len(d), 300_000), random_state=SEED)
rho = sm.groupby(["phase", "maia_elo_cat_self"]).apply(lambda x: x[["time_spent", "maia_entropy_norm"]].corr(method="spearman").iloc[0, 1] if len(x) > 500 else np.nan)
display(rho.unstack().reindex(columns=ELO_ORDER).round(3).rename_axis("Spearman rho(think-time, Maia entropy)"))

t_ = d.groupby("think_bin").agg(entropy=("maia_entropy_norm", "mean"), p_true=("maia_p_true", "mean")).reindex(THINK)
rf = cluster_rates(d, "think_bin").reindex(THINK)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=THINK, y=rf["rate"], mode="lines+markers", name="failure rate", line=dict(color=C_HUMAN),
              error_y=dict(type="data", symmetric=False, array=rf["hi"] - rf["rate"], arrayminus=rf["rate"] - rf["lo"])), secondary_y=False)
fig.add_trace(go.Scatter(x=THINK, y=t_["entropy"], mode="lines+markers", name="Maia normalised entropy", line=dict(color=C_MAIA)), secondary_y=True)
fig.update_yaxes(title_text="failure rate", secondary_y=False); fig.update_yaxes(title_text="Maia uncertainty", secondary_y=True)
fig.update_layout(height=420, title="Human think-time vs failure and vs Maia's own uncertainty")
register(fig, "Clock & history", "Think-time vs failure and uncertainty", "If the blue line is flat while the red line moves, Maia's uncertainty is blind to what makes the human deliberate.")

maia_elo_cat_self,<1100,1100-1199,1200-1299,1300-1399,1400-1499,1500-1599,1600-1699,1700-1799,1800-1899,1900-1999,>=2000
"Spearman rho(think-time, Maia entropy)",,,,,,,,,,,
endgame,0.175,0.175,0.175,0.172,0.185,0.162,0.198,0.203,0.197,0.192,0.134
middlegame,0.184,0.232,0.244,0.260,0.261,0.253,0.290,0.274,0.281,0.270,0.270
opening,-0.060,-0.106,-0.079,-0.089,-0.099,-0.097,-0.092,-0.087,-0.059,-0.050,0.000


In [17]:
QO = QUAL
fig = make_subplots(1, 3, subplot_titles=["recapture?", "opponent's previous move", "player's own previous move"], shared_yaxes=True)
for i, (col, order) in enumerate([("is_recapture", ["False", "True"]), ("opp_prev_quality", QO), ("own_prev_quality", QO)], start=1):
    r = cluster_rates(d, col).reindex(order)
    fig.add_trace(go.Bar(x=order, y=r["rate"], marker_color=C_MAIA, showlegend=False,
                         error_y=dict(type="data", symmetric=False, array=r["hi"] - r["rate"], arrayminus=r["rate"] - r["lo"]),
                         customdata=r["n"], hovertemplate="%{x}<br>failure %{y:.3f}<br>n=%{customdata:,}<extra></extra>"), 1, i)
    fig.add_hline(y=GLOBAL_FAIL, line_dash="dash", line_color="black", row=1, col=i)
fig.update_layout(height=420, yaxis_title="failure rate", title="History effects Maia cannot see (95% CI by game)")
register(fig, "Clock & history", "History effects", "A recapture is visible on the board only as a material imbalance; 'tilt' after a blunder is invisible. Do the bars differ from the dashed overall rate?")

# is the recapture/history effect just a confound of phase / think-time?  quick check: stratify
display(cluster_rates(d, ["phase", "is_recapture"]).round(3))

,phase,is_recapture,n,rate,lo,hi
opening|False,opening,False,221468,0.589,0.587,0.591
opening|True,opening,True,17988,0.135,0.130,0.139
middlegame|False,middlegame,False,247317,0.530,0.528,0.533
middlegame|True,middlegame,True,31793,0.152,0.147,0.156
endgame|True,endgame,True,8759,0.093,0.086,0.099
endgame|False,endgame,False,142434,0.469,0.465,0.472


## 5. Does the Elo input do what it should? (counterfactual experiment – re-runs Maia)
Maia-2 is *conditioned* on both players' Elo. We feed the **same** positions with every possible assumed rating and check
1. **Self-Elo:** is the log-likelihood of the human move highest when the assumed rating = true rating? (A heatmap with the peak on the diagonal means the conditioning works; a smeared or absent peak means it is mostly decorative.) Edge categories (`<1100`, `>=2000`) are clamped – what happens there?
2. **Opponent-Elo:** how much does the opponent's rating change predictions?
3. **Elo-sensitivity per position:** Jensen–Shannon divergence between the prediction for a 1100 and a 1900 player. *Where* does Maia think skill matters, and is that where it fails?

Cost: ~2×11 forward passes per sampled position. Reduce `ELO_N` on a CPU.

In [18]:
import torch
from maia2 import model as maia_model
from maia2.inference import prepare
from maia2.utils import board_to_tensor, map_to_category, mirror_move

ELO_N = 400                                  # positions per (time control × Elo category)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
nets = {tc: maia_model.from_pretrained(type=tc, device=device) for tc in ("blitz", "rapid")}
for _n in nets.values(): _n.eval()              # BatchNorm/dropout must be in inference mode, otherwise predictions differ from the parquet
all_moves_dict, elo_dict, all_moves_dict_reversed = prepare()
EL_LAB = list(elo_dict.keys()); EL_VAL = [elo_dict[l] for l in EL_LAB]; K = len(EL_VAL)
LO, HI = 1, K - 2                            # 1100-1199 vs 1900-1999 for the sensitivity measure

class BoardOnly(torch.utils.data.Dataset):
    def __init__(self, fens): self.fens = list(fens)
    def __len__(self): return len(self.fens)
    def __getitem__(self, i):
        board = chess.Board(self.fens[i])
        if board.turn == chess.BLACK: board = board.mirror()
        legal = torch.zeros(len(all_moves_dict)); legal[[all_moves_dict[m.uci()] for m in board.legal_moves]] = 1
        return board_to_tensor(board), legal

def jsd(p, q):
    m = 0.5 * (p + q)
    kl = lambda x, y: (x * (torch.log(x.clamp_min(1e-12)) - torch.log(y.clamp_min(1e-12)))).sum(1)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

@torch.no_grad()
def elo_experiment(sub, net, batch=256):
    is_black = sub["fen"].str.split(" ").str[1].eq("b").to_numpy()
    true_idx = np.array([all_moves_dict[mirror_move(m) if b else m] for m, b in zip(sub["move"], is_black)])
    cs = np.array([map_to_category(e, elo_dict) for e in sub["elo_self"]]); co = np.array([map_to_category(e, elo_dict) for e in sub["elo_oppo"]])
    n_ = len(sub); out = {n: np.zeros((n_, K)) for n in ("lp_self", "acc_self", "lp_opp", "acc_opp")}
    out["jsd_self"] = np.zeros(n_); out["jsd_opp"] = np.zeros(n_)
    loader = torch.utils.data.DataLoader(BoardOnly(sub["fen"]), batch_size=batch, shuffle=False, num_workers=0)
    off = 0
    for boards, legal in tqdm(loader):
        n = boards.shape[0]; sl = slice(off, off + n)
        boards, legal = boards.to(device), legal.to(device).bool()
        t = torch.as_tensor(true_idx[sl], device=device); cs_t = torch.as_tensor(cs[sl], device=device); co_t = torch.as_tensor(co[sl], device=device)
        for tag in ("self", "opp"):
            keep = {}
            for j, v in enumerate(EL_VAL):
                full = torch.full_like(cs_t, v)
                logits = net(boards, full, co_t)[0] if tag == "self" else net(boards, cs_t, full)[0]
                pr = logits.masked_fill(~legal, float("-inf")).softmax(-1)
                out[f"lp_{tag}"][sl, j] = pr.gather(1, t[:, None]).clamp_min(1e-9).log().squeeze(1).cpu().numpy()
                out[f"acc_{tag}"][sl, j] = (pr.argmax(1) == t).float().cpu().numpy()
                if j in (LO, HI): keep[j] = pr
            out[f"jsd_{tag}"][sl] = jsd(keep[LO], keep[HI]).cpu().numpy()
        off += n
    res = pd.DataFrame({**{f"{k_}_{j}": v[:, j] for k_, v in out.items() if v.ndim == 2 for j in range(K)},
                        "jsd_self": out["jsd_self"], "jsd_opp": out["jsd_opp"], "cat_self": cs, "cat_opp": co}, index=sub.index)
    return res

parts = []
for tc, sub in d.groupby("time_control_type"):
    s = sub.groupby("maia_elo_cat_self", group_keys=False).apply(lambda x: x.sample(min(len(x), ELO_N), random_state=SEED))
    parts.append(elo_experiment(s, nets[tc]).join(s[["failed", "phase", "move_number", "game_id", "maia_p_true", "time_control_type", "maia_elo_cat_self"]]))
ex = pd.concat(parts)
ex["true_j"] = ex["cat_self"].map({v: j for j, v in enumerate(EL_VAL)}); ex["true_jo"] = ex["cat_opp"].map({v: j for j, v in enumerate(EL_VAL)})
print(len(ex), "experiment rows")

Model for blitz games loaded to cpu.
Model for rapid games loaded to cpu.


  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

8800 experiment rows


In [19]:
# sanity check 4 (do it!): the log-likelihood at the TRUE Elo category must reproduce the parquet's maia_p_true
L_ = ex[[f"lp_self_{j}" for j in range(K)]].to_numpy(); tj_ = ex["true_j"].to_numpy().astype(int)
print("max |exp(lp at true Elo) - maia_p_true| =", np.abs(np.exp(L_[np.arange(len(L_)), tj_]) - ex["maia_p_true"].to_numpy()).max().round(5), "(should be ~1e-3 or less)")

max |exp(lp at true Elo) - maia_p_true| = 0.0 (should be ~1e-3 or less)


In [20]:
cols = lambda tag, m: [f"{m}_{tag}_{j}" for j in range(K)]
def true_vs_assumed(tag, metric, true_col):
    M = np.vstack([ex.loc[ex[true_col] == j, cols(tag, metric)].mean().to_numpy() if (ex[true_col] == j).any() else np.full(K, np.nan) for j in range(K)])
    return M

M_lp = true_vs_assumed("self", "lp", "true_j"); M_acc = true_vs_assumed("self", "acc", "true_j")
M_lp_rel = M_lp - np.nanmean(M_lp, axis=1, keepdims=True)
fig = make_subplots(1, 2, subplot_titles=["Δ log-likelihood of human move (vs. row mean)", "top-1 accuracy"], horizontal_spacing=0.14)
fig.add_trace(go.Heatmap(z=M_lp_rel, x=EL_LAB, y=EL_LAB, colorscale="RdBu", zmid=0, colorbar=dict(x=0.43, title="Δ log p")), 1, 1)
fig.add_trace(go.Heatmap(z=M_acc, x=EL_LAB, y=EL_LAB, colorscale="Viridis", colorbar=dict(title="acc")), 1, 2)
for c in (1, 2):
    fig.add_trace(go.Scatter(x=EL_LAB, y=EL_LAB, mode="lines", line=dict(color="black", dash="dash"), showlegend=False), 1, c)
    fig.update_xaxes(title_text="Elo category fed to Maia", row=1, col=c)
fig.update_yaxes(title_text="player's TRUE Elo category", row=1, col=1)
fig.update_layout(height=520, title="Counterfactual Elo: does Maia fit best when told the true rating? (dashed = diagonal)")
register(fig, "Elo response", "Assumed vs true Elo", "A sharp ridge on the diagonal = Elo conditioning works. Look separately at the first and last rows/columns (clamped categories).")

# per-row 'Elo gain': log-likelihood at the true category minus the mean over all assumed categories
L_ = ex[cols("self", "lp")].to_numpy(); tj_ = ex["true_j"].to_numpy().astype(int)
ex["elo_gain"] = L_[np.arange(len(L_)), tj_] - L_.mean(axis=1)
print("mean Elo gain (nats per move):", ex["elo_gain"].mean().round(4))
display(ex.groupby("maia_elo_cat_self")["elo_gain"].mean().reindex(ELO_ORDER).round(4).to_frame("mean Elo gain"))

# opponent Elo
fig = go.Figure()
for j in range(K):
    sub = ex[ex["true_j"] == j]
    if len(sub) < 50: continue
    m_ = sub[cols("opp", "lp")].mean().to_numpy(); fig.add_trace(go.Scatter(x=EL_LAB, y=m_ - m_.mean(), mode="lines", name=f"self {EL_LAB[j]}"))
fig.update_layout(height=420, title="Opponent Elo: change in log-likelihood when the assumed OPPONENT rating varies", xaxis_title="Elo category fed as opponent", yaxis_title="Δ log p (vs. mean)")
register(fig, "Elo response", "Opponent Elo effect", "Curves much flatter than the self-Elo ridge ⇒ the opponent's rating barely matters to Maia.")

mean Elo gain (nats per move): 0.0459


,mean Elo gain
maia_elo_cat_self,
<1100,0.0616
1100-1199,0.0578
1200-1299,0.0343
1300-1399,0.0526
1400-1499,0.0331
1500-1599,0.0282
1600-1699,0.0438
1700-1799,0.0368
1800-1899,0.0491


In [21]:
# Elo-sensitivity per position and its relation to failure
ex["sens_q"] = pd.qcut(ex["jsd_self"].rank(method="first"), 5, labels=["Q1 insensitive", "Q2", "Q3", "Q4", "Q5 very sensitive"])
fig = make_subplots(1, 2, subplot_titles=["Elo-sensitivity (JSD 1100 vs 1900) by phase", "failure rate by Elo-sensitivity quintile"])
for ph in PHASES:
    fig.add_trace(go.Box(y=ex.loc[ex["phase"] == ph, "jsd_self"], name=ph, boxpoints=False), 1, 1)
r = cluster_rates(ex, "sens_q").reindex([str(c) for c in ex["sens_q"].cat.categories])
fig.add_trace(go.Bar(x=r.index, y=r["rate"], marker_color=C_MAIA, showlegend=False, error_y=dict(type="data", symmetric=False, array=r["hi"] - r["rate"], arrayminus=r["rate"] - r["lo"])), 1, 2)
fig.update_layout(height=430, showlegend=False, title="Where does Maia believe skill matters – and does it fail there?")
register(fig, "Elo response", "Elo-sensitivity per position", "Left: positions where a 1100 and a 1900 player are predicted to behave differently. Right: failure rate across sensitivity quintiles.")

## 6. Style bias – what *kind* of move does Maia prefer, compared with humans?
Earlier we saw "Maia over-predicts knight moves" – but only inside **confident misses**, which proves nothing about bias (a miss population is selected on disagreement). Here we compare the **move-type mix of Maia's top-1 predictions with the human moves over all positions**, with game-clustered CIs, per phase and per Elo category. Per feature, a *paired* difference (Maia rate − human rate) is bootstrapped, so a CI excluding 0 is a genuine, systematic bias.

In [22]:
FEATS = {"pawn move": ("piece", "P"), "knight move": ("piece", "N"), "bishop move": ("piece", "B"), "rook move": ("piece", "R"), "queen move": ("piece", "Q"), "king move": ("piece", "K"),
         "capture": ("capture", True), "gives check": ("check", True), "castling": ("castle", True), "promotion": ("promo", True),
         "forward move": ("dir", 1), "backward move": ("dir", -1), "sideways move": ("dir", 0)}
for nm, (col, val) in FEATS.items():
    d[f"hf_{nm}"] = (d[f"h_{col}"] == val).astype(int); d[f"mf_{nm}"] = (d[f"m_{col}"] == val).astype(int); d[f"df_{nm}"] = d[f"mf_{nm}"] - d[f"hf_{nm}"]

fig = make_subplots(1, 3, shared_yaxes=True, subplot_titles=[p.capitalize() for p in PHASES], horizontal_spacing=0.03)
tab_rows = []
for ci, ph in enumerate(PHASES, start=1):
    sub = d[d["phase"] == ph]; xs_h, xs_m, ys, hov = [], [], [], []
    for nm in FEATS:
        h = cluster_rates(sub.assign(_g="all"), "_g", f"hf_{nm}").iloc[0]; m = cluster_rates(sub.assign(_g="all"), "_g", f"mf_{nm}").iloc[0]
        dd = cluster_rates(sub.assign(_g="all"), "_g", f"df_{nm}").iloc[0]
        sig = "significant" if (dd["lo"] > 0 or dd["hi"] < 0) else "within CI"
        xs_h.append(h["rate"]); xs_m.append(m["rate"]); ys.append(nm)
        hov.append(f"{nm}<br>human {h['rate']:.3f} | Maia {m['rate']:.3f}<br>diff {dd['rate']:+.3f} [{dd['lo']:+.3f}, {dd['hi']:+.3f}] ({sig})")
        tab_rows.append(dict(phase=ph, feature=nm, human=h["rate"], maia=m["rate"], diff=dd["rate"], lo=dd["lo"], hi=dd["hi"]))
    lx, ly = [], []
    for a_, b_, y_ in zip(xs_h, xs_m, ys): lx += [a_, b_, None]; ly += [y_, y_, None]
    fig.add_trace(go.Scatter(x=lx, y=ly, mode="lines", line=dict(color="#c9ced6"), showlegend=False, hoverinfo="skip"), 1, ci)
    fig.add_trace(go.Scatter(x=xs_h, y=ys, mode="markers", marker=dict(color=C_HUMAN, size=9), name="human", showlegend=(ci == 1), text=hov, hoverinfo="text"), 1, ci)
    fig.add_trace(go.Scatter(x=xs_m, y=ys, mode="markers", marker=dict(color=C_MAIA, size=9), name="Maia top-1", showlegend=(ci == 1), text=hov, hoverinfo="text"), 1, ci)
fig.update_layout(height=560, title="Move-type mix over ALL positions: humans vs Maia's top-1 (hover for paired CI of the difference)")
fig.update_xaxes(title_text="share of moves")
register(fig, "Move style", "Style: human vs Maia", "Gap between the red and blue dot = systematic bias of Maia's top-1 move. Hover shows whether the paired difference excludes 0.")
style_tab = pd.DataFrame(tab_rows); display(style_tab[(style_tab["lo"] > 0) | (style_tab["hi"] < 0)].round(4))

,phase,feature,human,maia,diff,lo,hi
0,opening,pawn move,0.4018,0.2989,-0.1029,-0.1047,-0.1006
1,opening,knight move,0.2443,0.3329,0.0886,0.0868,0.0905
2,opening,bishop move,0.1991,0.2034,0.0043,0.0026,0.0059
3,opening,rook move,0.0196,0.0166,-0.0030,-0.0036,-0.0023
4,opening,queen move,0.0758,0.0699,-0.0059,-0.0067,-0.0050
5,opening,king move,0.0593,0.0782,0.0189,0.0179,0.0199
6,opening,capture,0.1800,0.2164,0.0364,0.0348,0.0378
7,opening,gives check,0.0245,0.0277,0.0032,0.0026,0.0038
8,opening,castling,0.0507,0.0700,0.0193,0.0183,0.0202
10,opening,forward move,0.8628,0.8458,-0.0170,-0.0184,-0.0156


In [23]:
# does the bias depend on Elo?  (Maia rate - human rate, blank where the CI includes 0)
Z = np.full((len(FEATS), len(ELO_ORDER)), np.nan); T = np.full(Z.shape, "", dtype=object)
for i, nm in enumerate(FEATS):
    r = cluster_rates(d, "maia_elo_cat_self", f"df_{nm}").reindex(ELO_ORDER)
    for j, e in enumerate(ELO_ORDER):
        if e in r.index and r.loc[e, "n"] >= 500:
            Z[i, j] = r.loc[e, "rate"]; T[i, j] = f"{r.loc[e, 'rate']:+.3f}" if (r.loc[e, "lo"] > 0 or r.loc[e, "hi"] < 0) else "·"
zmax = np.nanmax(np.abs(Z))
fig = go.Figure(go.Heatmap(z=Z, x=ELO_ORDER, y=list(FEATS), text=T, texttemplate="%{text}", colorscale="RdBu", zmid=0, zmin=-zmax, zmax=zmax,
                           colorbar=dict(title="Maia − human"), hovertemplate="%{y}<br>%{x}<br>Maia − human = %{z:+.4f}<extra></extra>"))
fig.update_layout(height=520, title="Style bias by player Elo (· = difference not distinguishable from 0)", xaxis_title="player Elo category")
register(fig, "Move style", "Style bias by Elo", "Red = Maia predicts this move type more often than humans play it. A bias that grows with Elo would mean Maia is 'too strong-style' for weak players.")

# what does Maia predict instead of what the human played? (misses only)
CLS = ["P", "Px", "N", "Nx", "B", "Bx", "R", "Rx", "Q", "Qx", "K", "Kx", "castle", "promotion"]
PAL = ["#8da0cb", "#5b6fa8", "#66c2a5", "#2f8f78", "#fc8d62", "#d95f02", "#e78ac3", "#b03a8a", "#ffd92f", "#c9a800", "#a6a6a6", "#6e6e6e", "#a6d854", "#e5c494"]
miss = d[d["failed"]]
fig = go.Figure(); n_c = len(CLS)
for vi, ph in enumerate(["all"] + PHASES):
    s = miss if ph == "all" else miss[miss["phase"] == ph]
    ct = pd.crosstab(s["m_class"], s["h_class"]).reindex(index=CLS, columns=CLS).fillna(0)
    src, tgt, val = [], [], []
    for i, a_ in enumerate(CLS):
        for j, b_ in enumerate(CLS):
            if ct.iloc[i, j] > 0: src.append(i); tgt.append(n_c + j); val.append(float(ct.iloc[i, j]))
    fig.add_trace(go.Sankey(visible=(vi == 0), node=dict(label=[f"{c} (Maia)" for c in CLS] + [f"{c} (human)" for c in CLS], color=PAL + PAL, pad=12),
                            link=dict(source=src, target=tgt, value=val, color=["rgba(150,150,150,0.35)"] * len(src))))
fig.update_layout(height=620, title="Misses only: the move class Maia predicted → the move class the human played",
                  updatemenus=[dict(type="dropdown", x=0, y=1.12, xanchor="left", buttons=[dict(label=ph, method="update", args=[{"visible": [j == vi for j in range(4)]}]) for vi, ph in enumerate(["all"] + PHASES)])])
register(fig, "Move style", "Maia-predicted vs human move class (misses)", "Wide flows off the 'same class' diagonal show what Maia confuses (e.g. predicts a knight move where humans push a pawn). Switch phase with the dropdown.")

## 7. Failure atlas – which *conditions* predict failure?
Two complementary tools, both restricted to **conditions** (position, player, clock – *not* Maia's own outputs, which would be circular):

1. **Slice discovery:** a shallow decision tree. Every node is an interpretable slice (*"endgame, Elo < 1350, clock < 40 s"*) with its failure rate and a game-clustered CI. Rendered as an icicle chart; **in the dashboard, clicking a node shows real board examples** (blue arrow = Maia, red = human, confident misses first).
2. **Predictive check:** gradient boosting under *game-grouped* cross-validation. AUC with conditions only (A) vs. conditions + Maia's own confidence (B) says how much of the failure is predictable from the situation and how much Maia "knows" itself. Permutation importance ranks the conditions.

Set `ATLAS_FRAME/ATLAS_TARGET` to look at **real Maia errors** (from §2) instead of top-1 misses.

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

POS_FEATS = ["move_number", "elo_self", "elo_diff", "clock_before", "clock_frac", "time_spent", "n_legal", "non_pawn_material", "mat_balance",
             "n_pieces", "queens_on", "in_check", "is_recapture", "wc_before_f", "has_eval", "side_white", "is_rapid"]
BINARY = {"queens_on", "in_check", "is_recapture", "has_eval", "side_white", "is_rapid"}
PRETTY = dict(move_number="move no.", elo_self="player Elo", elo_diff="Elo − opp. Elo", clock_before="clock (s)", clock_frac="clock / base time", time_spent="think-time (s)",
              n_legal="legal moves", non_pawn_material="non-pawn material", mat_balance="material balance", n_pieces="pieces on board", queens_on="queens on",
              in_check="in check", is_recapture="recapture", wc_before_f="engine win-chance", has_eval="has engine eval", side_white="White to move", is_rapid="rapid game")

def make_atlas_frame(df):
    A = df.copy(); A["orig_idx"] = df.index
    A["wc_before_f"] = A["wc_before"].fillna(0); A["has_eval"] = A["wc_before"].notna().astype(int)
    A["side_white"] = (A["side"] == "white").astype(int); A["is_rapid"] = (A["time_control_type"] == "rapid").astype(int)
    for c in ("queens_on", "in_check", "is_recapture"): A[c] = A[c].astype(int)
    return A.reset_index(drop=True)

ATLAS_FRAME, ATLAS_TARGET = d, "failed"                 # alternative (after §2):  ATLAS_FRAME, ATLAS_TARGET = a, "maia_error"
A = make_atlas_frame(ATLAS_FRAME); A[ATLAS_TARGET] = A[ATLAS_TARGET].astype(int)
A["fail_cat"] = A["orig_idx"].map(a["fail_cat"]) if a is not None else np.nan          # engine-based failure category (sample only)
A["fail_kind"] = A["orig_idx"].map(k["fail_kind"])                                        # consensus-based kind (repeated positions only)
A["keyed"] = A["orig_idx"].isin(k.index)
GROUPS = A["game_id"].to_numpy()

# ---- predictive check (game-grouped CV) ----
As = A.sample(min(len(A), 300_000), random_state=SEED).reset_index(drop=True)
def cv_auc(feats, n_splits=5):
    aucs, last = [], None
    for tr, te in GroupKFold(n_splits).split(As, groups=As["game_id"]):
        m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, random_state=SEED).fit(As.loc[tr, feats], As.loc[tr, ATLAS_TARGET])
        aucs.append(roc_auc_score(As.loc[te, ATLAS_TARGET], m.predict_proba(As.loc[te, feats])[:, 1])); last = (m, te)
    return np.array(aucs), last
auc_A, (mA, te) = cv_auc(POS_FEATS)
auc_B, _ = cv_auc(POS_FEATS + ["maia_p_top1", "maia_entropy_norm"])
print(f"AUC conditions only (A): {auc_A.mean():.3f} ± {auc_A.std():.3f} | + Maia confidence (B): {auc_B.mean():.3f} ± {auc_B.std():.3f}")

pi = permutation_importance(mA, As.loc[te[:60000], POS_FEATS], As.loc[te[:60000], ATLAS_TARGET], scoring="roc_auc", n_repeats=3, random_state=SEED)
imp = pd.DataFrame({"feature": [PRETTY[f] for f in POS_FEATS], "mean": pi.importances_mean, "std": pi.importances_std}).sort_values("mean")
fig = go.Figure(go.Bar(x=imp["mean"], y=imp["feature"], orientation="h", error_x=dict(type="data", array=imp["std"]), marker_color=C_MAIA))
fig.update_layout(height=520, title=f"What predicts failure? Permutation importance (AUC drop), conditions-only model, AUC = {auc_A.mean():.3f}; with Maia confidence {auc_B.mean():.3f}",
                  xaxis_title="drop in AUC when the feature is shuffled")
register(fig, "Failure atlas", "Condition importance", "Game-grouped CV. If AUC(A) is near 0.5, failures are not explained by simple conditions and the interesting structure is in the slices below.")

AUC conditions only (A): 0.697 ± 0.002 | + Maia confidence (B): 0.771 ± 0.003


In [25]:
# adjusted vs raw effect: is a raw pattern (e.g. 'fast moves fail more') just a confound of phase / Elo / legal moves?
from sklearn.inspection import partial_dependence
PD_FEATS = ["time_spent", "clock_before", "n_legal", "elo_self", "move_number", "mat_balance"]
Xs = As.loc[te[:30000], POS_FEATS].astype(float)
fig = make_subplots(2, 3, subplot_titles=[PRETTY[f] for f in PD_FEATS], vertical_spacing=0.16)
for i, f in enumerate(PD_FEATS):
    rr, cc = i // 3 + 1, i % 3 + 1
    pdr = partial_dependence(mA, Xs, [f], grid_resolution=25, kind="average")
    gx = pdr.get("grid_values", pdr.get("values"))[0]
    fig.add_trace(go.Scatter(x=gx, y=pdr["average"][0], mode="lines", line=dict(color=C_MAIA), name="adjusted (partial dependence)", showlegend=(i == 0)), rr, cc)
    qb = pd.qcut(As[f], 25, duplicates="drop")
    raw = As.groupby(qb, observed=True).agg(x=(f, "mean"), y=(ATLAS_TARGET, "mean"))
    fig.add_trace(go.Scatter(x=raw["x"], y=raw["y"], mode="lines", line=dict(color=C_HUMAN, dash="dot"), name="raw marginal rate", showlegend=(i == 0)), rr, cc)
fig.update_layout(height=640, title="Adjusted (blue) vs raw (red dotted) effect of each condition on failure probability")
register(fig, "Failure atlas", "Adjusted vs raw effects", "Where blue and red diverge, the raw pattern is (partly) a confound: the other conditions, not this one, explain it. Where they agree, the effect is robust to adjustment.")

In [26]:
import numpy as np

# Check for infinities
inf_cols = A[POS_FEATS].columns[np.isinf(A[POS_FEATS]).any()]
print("Columns with infinities:", list(inf_cols))

inf_count = np.isinf(A['clock_frac']).sum()
print(f"Number of infinite values in 'clock_frac': {inf_count}")

A['clock_frac'] = A['clock_frac'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

Columns with infinities: ['clock_frac']
Number of infinite values in 'clock_frac': 698


In [27]:
TREE_DEPTH = 4
MIN_LEAF = max(1000, int(0.002 * len(A)))
tree = DecisionTreeClassifier(max_depth=TREE_DEPTH, min_samples_leaf=MIN_LEAF, random_state=SEED).fit(A[POS_FEATS].to_numpy(), A[ATLAS_TARGET])
ATREE = tree.tree_
path = tree.decision_path(A[POS_FEATS].to_numpy()).tocsc()
rows_of = lambda nid: path.indices[path.indptr[nid]:path.indptr[nid + 1]]
y_all = A[ATLAS_TARGET].to_numpy(float); gcode_all = pd.factorize(A["game_id"])[0]; glob = y_all.mean()

def fmt_cond(f, op, th):
    if f in BINARY: return f"{PRETTY[f]} = {'yes' if (op == '>') else 'no'}"
    th_s = f"{th:.0f}" if abs(th) >= 10 else f"{th:.2f}"
    return f"{PRETTY[f]} {op} {th_s}"

cond, parent = {0: "all positions"}, {0: ""}
for nid in range(ATREE.node_count):
    l, r_ = ATREE.children_left[nid], ATREE.children_right[nid]
    if l != -1:
        f, th = POS_FEATS[ATREE.feature[nid]], ATREE.threshold[nid]
        cond[l], cond[r_] = fmt_cond(f, "≤", th), fmt_cond(f, ">", th); parent[l] = parent[r_] = str(nid)

def path_text(nid):
    out = []
    while nid != 0:
        out.append(cond[nid]); nid = int(parent[nid])
    return " → ".join(reversed(out)) or "all positions"

def board_svg(fen, human_uci, maia_uci, size=230):
    b = chess.Board(fen)
    arrows = [chess.svg.Arrow(chess.Move.from_uci(maia_uci).from_square, chess.Move.from_uci(maia_uci).to_square, color="#2a78d6cc"),
              chess.svg.Arrow(chess.Move.from_uci(human_uci).from_square, chess.Move.from_uci(human_uci).to_square, color="#d9534fcc")]
    return chess.svg.board(b, arrows=arrows, size=size, orientation=b.turn, coordinates=False)

def node_cards(sub, n_conf=3, n_rand=3):
    f = sub[sub[ATLAS_TARGET] == 1]
    if f.empty: return []
    conf = f.nlargest(n_conf, "maia_p_top1"); rest = f.drop(conf.index); rest = rest.sample(min(n_rand, len(rest)), random_state=SEED)
    cards = []
    for tag, grp in (("confident miss", conf), ("random miss", rest)):
        for _, r in grp.iterrows():
            cap = (f"<b>{tag}</b> · Elo {int(r['elo_self'])} vs {int(r['elo_oppo'])} · move {int(r['move_number'])} · thought {r['time_spent']:.0f}s<br>"
                   f"<span style='color:#2a78d6'>Maia {r['maia_move']} ({r['maia_p_top1']:.0%})</span> · <span style='color:#d9534f'>human {r['move']} (Maia gave it {r['maia_p_true']:.1%})</span>")
            cards.append(f"<div class='card'>{board_svg(r['fen'], r['move'], r['maia_move'])}<div class='cap'>{cap}</div></div>")
    return cards

NODES, ids, labels, parents, values, colors, custom = {}, [], [], [], [], [], []
for nid in tqdm(range(ATREE.node_count), desc="atlas nodes"):
    idx = rows_of(nid); sub = A.iloc[idx]; rate = y_all[idx].mean(); lo, hi = ci_idx(gcode_all, y_all, idx)
    qd = sub.dropna(subset=["move_quality"])
    stats = dict(n=int(len(idx)), rate=float(rate), lo=float(lo), hi=float(hi), lift=float(rate / glob), top3_miss=float(1 - sub["maia_top3_hit"].mean()),
                 mean_conf=float(sub["maia_p_top1"].mean()), mean_entropy=float(sub["maia_entropy_norm"].mean()),
                 human_blunder=float((qd["move_quality"] == "blunder").mean()) if len(qd) else None, median_think=float(sub["time_spent"].median()))
    NODES[str(nid)] = dict(path=path_text(nid), stats=stats, cards=node_cards(sub), target=ATLAS_TARGET)
    ids.append(str(nid)); parents.append(parent[nid]); values.append(len(idx)); colors.append(rate)
    labels.append(f"{cond[nid]}<br>{rate:.0%}"); custom.append([len(idx), rate, lo, hi, rate / glob])

fig = go.Figure(go.Icicle(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total",
                          marker=dict(colors=colors, colorscale="RdBu_r", cmid=glob, colorbar=dict(title="rate")), customdata=custom,
                          hovertemplate="%{label}<br>n=%{customdata[0]:,}<br>rate %{customdata[1]:.3f} [%{customdata[2]:.3f}, %{customdata[3]:.3f}]<br>lift ×%{customdata[4]:.2f}<extra></extra>",
                          tiling=dict(orientation="v", pad=2), root_color="#dfe3e8"))
fig.update_layout(height=640, margin=dict(t=40, l=5, r=5, b=5), title=f"Failure atlas – target: '{ATLAS_TARGET}' (colour = rate, red above average)")
register(fig, "Failure atlas", "Atlas (static)", "In the dashboard this chart is linked to board examples.", show=True)
leaf_tab = pd.DataFrame([dict(path=NODES[i]["path"], **NODES[i]["stats"]) for i in ids if ATREE.children_left[int(i)] == -1]).sort_values("lift", ascending=False)
display(leaf_tab[["path", "n", "rate", "lo", "hi", "lift", "human_blunder", "mean_conf", "mean_entropy"]].round(3))

atlas nodes:   0%|          | 0/31 [00:00<?, ?it/s]

,path,n,rate,lo,hi,lift,human_blunder,mean_conf,mean_entropy
0,recapture = no → move no. ≤ 2.50 → move no. ≤ ...,16302,0.948,0.944,0.951,1.887,0.000,0.403,0.678
1,recapture = no → move no. ≤ 2.50 → move no. ≤ ...,3698,0.864,0.854,0.875,1.720,0.001,0.510,0.563
2,recapture = no → move no. ≤ 2.50 → move no. > ...,12975,0.742,0.734,0.749,1.477,0.002,0.376,0.629
7,recapture = no → move no. > 2.50 → think-time ...,89149,0.661,0.657,0.664,1.316,0.106,0.460,0.489
3,recapture = no → move no. ≤ 2.50 → move no. > ...,6807,0.633,0.621,0.645,1.260,0.003,0.318,0.668
6,recapture = no → move no. > 2.50 → think-time ...,137987,0.581,0.578,0.584,1.157,0.085,0.490,0.473
5,recapture = no → move no. > 2.50 → think-time ...,303874,0.471,0.469,0.473,0.938,0.051,0.531,0.425
4,recapture = no → move no. > 2.50 → think-time ...,40427,0.334,0.328,0.340,0.665,0.038,0.688,0.537
14,recapture = yes → think-time (s) > 3.50 → thin...,1751,0.300,0.279,0.321,0.597,0.067,0.691,0.273
15,recapture = yes → think-time (s) > 3.50 → thin...,4121,0.258,0.247,0.271,0.513,0.056,0.720,0.246


### 7.1 Slice dossier – *why* is a slice different? (shown in the dashboard when you click an atlas block)
For **every** slice the notebook computes
* **What distinguishes it** – standardised mean difference of each condition against all other positions, split into the *defining* conditions (trivially different) and **what else differs** (the interesting part: latent confounders);
* **What Maia does inside it** – Maia-minus-human move-type mix, over-confidence gap, share of confident misses, and, where available, the engine-based failure kind (§2) and the consensus-based kind (§3);
* **Why it holds this much data** – share of all positions, how many distinct positions it contains, how many are repeated openings, and how *compact* it is in feature space;
* **Sub-slices** – does it split into parts that fail at different rates and make *different kinds* of predictions? For leaves the best split is found on **half of the games and evaluated on the other half**, so a "difference" is not just the tree's selection effect.

In [28]:
X = A[POS_FEATS].to_numpy(float); N_ALL = len(A)
sd_all = X.std(0) + 1e-9; sum_all = X.sum(0)

def chain(nid):
    out = []
    while nid != 0: out.append(nid); nid = int(parent[nid])
    return out
def used_feats(nid): return {POS_FEATS[ATREE.feature[int(parent[x])]] for x in chain(nid)}

def bars_html(items, scale=1.0):
    rows = ""
    for name, v, txt in items:
        w = min(100, abs(v) / scale * 100); col = "#d9534f" if v > 0 else "#2a78d6"
        rows += f"<div class='brow'><span class='bl'>{name}</span><span class='bt'><i style='width:{w:.0f}%;background:{col}'></i></span><span class='bv'>{txt}</span></div>"
    return rows

def class_mix(sub, col): return sub[col].value_counts(normalize=True).reindex(CLS).fillna(0).to_numpy()
def jsd_np(p, q):
    from scipy.spatial.distance import jensenshannon
    return float(jensenshannon(p, q) ** 2)

def probe_html(nid):
    idx = rows_of(nid)
    if ATREE.children_left[nid] != -1:
        kids = [int(ATREE.children_left[nid]), int(ATREE.children_right[nid])]; kid_idx = [rows_of(c) for c in kids]; desc = [cond[c] for c in kids]
        note = "the atlas's own split"
    else:
        if len(idx) < 4000: return "<div class='sub'>too small to probe for sub-slices</div>"
        fit, test = idx[gcode_all[idx] % 2 == 0], idx[gcode_all[idx] % 2 == 1]
        st = DecisionTreeClassifier(max_depth=1, min_samples_leaf=max(300, int(0.1 * len(fit))), random_state=SEED).fit(X[fit], y_all[fit])
        if st.tree_.node_count == 1:
            return "<div class='sub'>no further split improves purity: the slice looks homogeneous in these conditions</div>"
        j, th = int(st.tree_.feature[0]), float(st.tree_.threshold[0])
        kid_idx = [test[X[test, j] <= th], test[X[test, j] > th]]; desc = [fmt_cond(POS_FEATS[j], "≤", th), fmt_cond(POS_FEATS[j], ">", th)]
        note = "best split found on half the games, rates measured on the OTHER half"
    rows = []
    for dsc, ii in zip(desc, kid_idx):
        if len(ii) < 30: continue
        lo, hi = ci_idx(gcode_all, y_all, ii); sb = A.iloc[ii]
        rows.append((dsc, len(ii), y_all[ii].mean(), lo, hi, class_mix(sb, "m_class"), class_mix(sb, "h_class")))
    if len(rows) < 2: return "<div class='sub'>too few positions on one side</div>"
    differ = rows[0][3] > rows[1][4] or rows[1][3] > rows[0][4]
    js_m, js_h = jsd_np(rows[0][5], rows[1][5]), jsd_np(rows[0][6], rows[1][6])
    tbl = "".join(f"<div class='sub'>• <b>{r[0]}</b>: n={r[1]:,}, rate {r[2]:.1%} [{r[3]:.1%}, {r[4]:.1%}]</div>" for r in rows)
    verdict = ("<b>sub-slices fail at different rates</b> (CIs do not overlap)" if differ else "failure rates not distinguishable between the parts")
    return (f"<div class='sub'>({note})</div>{tbl}<div class='sub'>{verdict}; move-type-mix divergence (JSD, &gt;0.02 = visibly different): "
            f"Maia {js_m:.3f} · human {js_h:.3f}</div>")

def dossier(nid):
    idx = rows_of(nid); sub = A.iloc[idx]; n = len(idx); used = used_feats(nid) if nid != 0 else set()
    smd = (X[idx].mean(0) - (sum_all - X[idx].sum(0)) / max(N_ALL - n, 1)) / sd_all
    if nid == 0: smd = np.zeros_like(smd)          # the root has no complement
    order = np.argsort(-np.abs(smd))
    fmt = lambda j: (PRETTY[POS_FEATS[j]], smd[j], f"{smd[j]:+.2f} sd")
    defining = [fmt(j) for j in order if POS_FEATS[j] in used][:4]; other = [fmt(j) for j in order if POS_FEATS[j] not in used][:5]
    tight = float(np.mean(X[idx].std(0) / sd_all))
    mm, hh = class_mix(sub, "m_class"), class_mix(sub, "h_class"); diff = mm - hh; top = np.argsort(-np.abs(diff))[:3]
    style = [(f"{CLS[j]}", diff[j], f"{diff[j] * 100:+.1f} pp") for j in top]
    fl = sub["failed"].astype(bool); gap = sub["maia_p_top1"].mean() - sub["maia_correct"].mean()
    conf_share = ((sub["maia_p_top1"] >= 0.6) & fl).sum() / max(fl.sum(), 1)
    out = ["<div class='dos'>"]
    if defining: out += ["<h4>Defining conditions (distance from the rest, in sd)</h4>", bars_html(defining, 1.5)]
    out += ["<h4>What ELSE differs from the rest of the data</h4>", bars_html(other, 1.0), "<div class='sub'>conditions the tree did not use: candidate explanations / confounders</div>"]
    out += ["<h4>What Maia does inside this slice</h4>", "<div class='sub'>Maia-minus-human share of moves, largest gaps:</div>", bars_html(style, 0.1),
            f"<div class='sub'>over-confidence gap (mean confidence − accuracy): <b>{gap:+.3f}</b> · share of misses that are confident (≥60%): <b>{conf_share:.0%}</b></div>"]
    sa = sub[fl & sub["fail_cat"].notna()]
    if len(sa) >= 30:
        sh = sa["fail_cat"].value_counts(normalize=True).reindex(CATS[1:]).fillna(0)
        out.append("<div class='sub'>engine check of misses (n=%d analysed): %s</div>" % (len(sa), " · ".join(f"{c.split(' (')[0]} {v:.0%}" for c, v in sh.items())))
    sk = sub[fl & sub["fail_kind"].notna()]
    if len(sk) >= 30:
        sh = sk["fail_kind"].value_counts(normalize=True)
        out.append("<div class='sub'>consensus check of misses (n=%d in repeated positions): %s</div>" % (len(sk), " · ".join(f"{c.split(' (')[0]} {v:.0%}" for c, v in sh.items())))
    out += ["<h4>Why this much data?</h4>",
            f"<div class='sub'>{n / N_ALL:.1%} of all positions · {sub['key'].nunique() / n:.0%} distinct positions · {sub['keyed'].mean():.0%} are repeated (opening-book) positions · "
            f"compactness {tight:.2f} (&lt;0.8 = tightly concentrated in condition space, ≈1 = as spread out as everything else)</div>",
            "<h4>Sub-slices: does Maia behave differently inside?</h4>", probe_html(nid), "</div>"]
    return "".join(out)

for nid in tqdm(range(ATREE.node_count), desc="dossiers"):
    NODES[str(nid)]["dossier"] = dossier(nid)
print("dossiers ready – they appear when you click a block in the dashboard's atlas")

dossiers:   0%|          | 0/31 [00:00<?, ?it/s]

dossiers ready – they appear when you click a block in the dashboard's atlas


## 7.2 Failure modes – are the misses one phenomenon or several?
The atlas slices *conditions*. Here we do the complementary thing: describe each **miss** by Maia's own *behavioural signature* – how confident it was, margin to its runner-up, entropy, the probability and rank it gave the human move, whether it picked the right piece, plus the human's think-time, legal-move count and move number – and **cluster the misses**. Then, for every cluster, the questions from the task:

* **What distinguishes it?** z-profile on the signature, and the contexts (phase, Elo, think-time, move class…) over-represented in it;
* **Why dense or sparse?** median distance to the 15 nearest neighbours in signature space (dense = many near-identical misses = a *systematic* mechanism; sparse = diverse, idiosyncratic misses) and within-cluster spread;
* **Sub-clusters?** a second clustering inside each cluster; the silhouette says whether there is real sub-structure or just a continuum, and the sub-clusters are compared on *what Maia predicts* (move-type mix) and on human blunder rate.

The 2-D map uses UMAP if installed (`pip install umap-learn`), otherwise PCA; map axes are not interpretable – only neighbourhoods are.

In [29]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

Mx = d[d["failed"]].copy(); Mx["orig_idx"] = Mx.index; Mx = Mx.reset_index(drop=True)
top2 = np.array([p[1] if len(p) > 1 else 0.0 for p in Mx["maia_top5_probs"]])
SIG = pd.DataFrame({"p_top1": Mx["maia_p_top1"].values, "margin": Mx["maia_p_top1"].values - top2, "entropy": Mx["maia_entropy_norm"].values,
                    "log_p_true": np.log10(np.clip(Mx["maia_p_true"].values, 1e-4, 1)), "log_rank_true": np.log2(Mx["maia_rank_true"].values),
                    "log_think": np.log1p(Mx["time_spent"].values), "log_legal": np.log(Mx["n_legal"].values), "move_no": Mx["move_number"].values,
                    "same_piece": (Mx["h_piece"].values == Mx["m_piece"].values).astype(float)})
Z = StandardScaler().fit_transform(SIG.to_numpy())
WORDS = {"p_top1": ("Maia confident", "Maia unsure"), "margin": ("clear favourite", "close call"), "entropy": ("diffuse distribution", "peaked distribution"),
         "log_p_true": ("human move plausible", "human move improbable"), "log_rank_true": ("human move ranked low", "human move ranked high"),
         "log_think": ("long think", "fast move"), "log_legal": ("many legal moves", "few legal moves"), "move_no": ("late game", "early game"),
         "same_piece": ("same piece, other target", "different piece")}

# choose k by silhouette (on a sample), but never fewer than 4 modes so the map stays informative
zs = Z[rng.choice(len(Z), min(20000, len(Z)), replace=False)]
sil = {kk: silhouette_score(zs, KMeans(kk, n_init=3, random_state=SEED).fit_predict(zs), sample_size=min(8000, len(zs)), random_state=SEED) for kk in range(4, 9)}
K_MODES = max(sil, key=sil.get); print("silhouette by k:", {kk: round(v, 3) for kk, v in sil.items()}, "-> k =", K_MODES)
km = KMeans(K_MODES, n_init=5, random_state=SEED).fit(Z)
raw_lab = km.labels_; size_rank = pd.Series(raw_lab).value_counts().index.tolist()
Mx["mode"] = pd.Series(raw_lab).map({c: i for i, c in enumerate(size_rank)}).values          # 0 = largest cluster

silhouette by k: {4: 0.202, 5: 0.205, 6: 0.18, 7: 0.166, 8: 0.17} -> k = 5


In [30]:
# density in signature space (kNN distance on a sample) + embedding
S = 30000; ix = rng.choice(len(Z), min(S, len(Z)), replace=False)
nn = NearestNeighbors(n_neighbors=16).fit(Z[ix]); dist, _ = nn.kneighbors(Z[ix]); knn = dist[:, 1:].mean(1)
try:
    import umap; emb = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=SEED).fit_transform(Z[ix]); EMB = "UMAP"
except Exception:
    emb = PCA(2, random_state=SEED).fit_transform(Z[ix]); EMB = "PCA"
smp = Mx.iloc[ix].copy(); smp["knn"] = knn; smp["x"], smp["y"] = emb[:, 0], emb[:, 1]

zprof = (SIG.groupby(Mx["mode"]).mean() - SIG.mean()) / SIG.std()
def mode_label(i):
    z = zprof.loc[i]; top = z.abs().sort_values(ascending=False).index[:2]
    return " & ".join(WORDS[f][0 if z[f] > 0 else 1] for f in top)
LAB = {i: f"M{i}: {mode_label(i)}" for i in range(K_MODES)}
Mx["mode_label"] = Mx["mode"].map(LAB); smp["mode_label"] = smp["mode"].map(LAB)

CTX = ["phase", "maia_elo_cat_self", "think_bin", "state", "h_class", "m_class", "move_quality"]
def ctx_lifts(mask, n_top=4):
    out = []
    for c in CTX:
        base = Mx[c].astype(str).where(Mx[c].notna()).value_counts(normalize=True); inn = Mx.loc[mask, c].astype(str).where(Mx.loc[mask, c].notna()).value_counts(normalize=True)
        for v, sh in inn.items():
            if sh >= 0.05 and base.get(v, 0) > 0: out.append((sh / base[v], f"{c}={v}", sh))
    return sorted(out, reverse=True)[:n_top]

# sub-clusters inside each mode
SUB, sub_sil = {}, {}
for i in range(K_MODES):
    idx = np.where(Mx["mode"].values == i)[0]
    if len(idx) < 3000: SUB[i] = np.zeros(len(idx), int); sub_sil[i] = np.nan; continue
    zi = Z[idx]; sidx = rng.choice(len(idx), min(15000, len(idx)), replace=False)
    best = max(((kk, silhouette_score(zi[sidx], KMeans(kk, n_init=3, random_state=SEED).fit_predict(zi[sidx]), sample_size=min(5000, len(sidx)), random_state=SEED)) for kk in (2, 3, 4)), key=lambda t: t[1])
    kmi = KMeans(best[0], n_init=3, random_state=SEED).fit(zi[sidx]); SUB[i] = kmi.predict(zi); sub_sil[i] = best[1]
Mx["sub"] = -1
for i in range(K_MODES): Mx.loc[Mx["mode"] == i, "sub"] = SUB[i]
print("sub-structure (silhouette; <0.15 ≈ continuum, >0.25 ≈ real sub-clusters):", {i: (None if np.isnan(v) else round(v, 2)) for i, v in sub_sil.items()})

sub-structure (silhouette; <0.15 ≈ continuum, >0.25 ≈ real sub-clusters): {0: 0.2, 1: 0.21, 2: 0.19, 3: 0.23, 4: 0.22}


In [31]:
def card_html(r, tag):
    cap = (f"<b>{tag}</b> · Elo {int(r['elo_self'])} vs {int(r['elo_oppo'])} · move {int(r['move_number'])} · thought {r['time_spent']:.0f}s<br>"
           f"<span style='color:#2a78d6'>Maia {r['maia_move']} ({r['maia_p_top1']:.0%})</span> · <span style='color:#d9534f'>human {r['move']} (Maia gave it {r['maia_p_true']:.1%})</span>")
    return f"<div class='card'>{board_svg(r['fen'], r['move'], r['maia_move'])}<div class='cap'>{cap}</div></div>"

MODES = {}; med_knn = np.median(knn)
for i in range(K_MODES):
    m = Mx["mode"].values == i; rows_i = Mx[m]; n_i = int(m.sum())
    s_i = smp[smp["mode"] == i]; rel_density = med_knn / max(np.median(s_i["knn"]), 1e-9) if len(s_i) else np.nan
    spread = float(np.linalg.norm(Z[m] - km.cluster_centers_[size_rank[i]], axis=1).mean())
    zi = zprof.loc[i].sort_values(key=np.abs, ascending=False)
    qd = rows_i.dropna(subset=["move_quality"]); bl = (qd["move_quality"] == "blunder").mean() if len(qd) else np.nan
    html = [f"<div class='dos'><b>{LAB[i]}</b><div class='sub'>{n_i:,} misses = {n_i / len(Mx):.1%} of all misses · human blunder rate {bl:.1%}</div>"
            "<h4>What distinguishes it (z-score of the signature vs. all misses)</h4>", bars_html([(f, v, f"{v:+.2f}") for f, v in zi.items()][:6], 2.0),
            "<h4>Contexts over-represented here (lift = share here / share among all misses)</h4>",
            "".join(f"<div class='sub'>• {name} · {sh:.0%} of this mode, ×{lift:.1f}</div>" for lift, name, sh in ctx_lifts(m)),
            "<h4>Dense or sparse?</h4>",
            f"<div class='sub'>relative density ×{rel_density:.2f} (1 = typical; &gt;1 means misses here are unusually similar to each other = a systematic mechanism; &lt;1 = diverse, idiosyncratic) · within-cluster spread {spread:.2f}</div>"]
    sa = rows_i[rows_i["orig_idx"].isin(a.index)] if a is not None else rows_i.iloc[0:0]
    if len(sa) >= 30:
        sh = a.loc[sa["orig_idx"], "fail_cat"].value_counts(normalize=True).reindex(CATS[1:]).fillna(0)
        html.append("<h4>Is it an error? (engine check, n=%d)</h4><div class='sub'>%s</div>" % (len(sa), " · ".join(f"{c.split(' (')[0]} {v:.0%}" for c, v in sh.items())))
    html.append("<h4>Sub-clusters</h4>")
    if np.isnan(sub_sil[i]): html.append("<div class='sub'>too small to subdivide</div>")
    else:
        kk = int(Mx.loc[m, "sub"].nunique()); verdict = "real sub-structure" if sub_sil[i] > 0.25 else ("weak sub-structure" if sub_sil[i] > 0.15 else "a continuum, no clear sub-clusters")
        html.append(f"<div class='sub'>{kk} sub-clusters, silhouette {sub_sil[i]:.2f} → {verdict}</div>")
        mixes = []
        for sc in sorted(rows_i["sub"].unique()):
            ms = rows_i["sub"].values == sc; zs_ = (SIG[m].reset_index(drop=True)[ms].mean() - SIG[m].mean()) / SIG[m].std()
            f2 = zs_.abs().sort_values(ascending=False).index[:2]; desc = " & ".join(WORDS[f][0 if zs_[f] > 0 else 1] for f in f2)
            qs = rows_i[ms].dropna(subset=["move_quality"]); b_ = (qs["move_quality"] == "blunder").mean() if len(qs) else np.nan
            mixes.append(class_mix(rows_i[ms], "m_class"))
            html.append(f"<div class='sub'>• sub {sc}: {ms.sum():,} misses · {desc} · human blunder rate {b_:.1%}</div>")
        if len(mixes) >= 2:
            pj = np.mean([jsd_np(mixes[p], mixes[q]) for p in range(len(mixes)) for q in range(p + 1, len(mixes))])
            html.append(f"<div class='sub'>mean divergence of what Maia predicts between sub-clusters (JSD) {pj:.3f}</div>")
    html.append("</div>")
    cen = km.cluster_centers_[size_rank[i]]; dd = np.linalg.norm(Z[m] - cen, axis=1); ordr = np.argsort(dd)
    cards = [card_html(rows_i.iloc[j], "typical") for j in ordr[:4]] + [card_html(rows_i.iloc[j], "random") for j in rng.choice(n_i, min(2, n_i), replace=False)]
    MODES[str(i)] = dict(html="".join(html), cards=cards)
print(f"{K_MODES} failure modes documented")

5 failure modes documented


In [32]:
PALM = ["#2a78d6", "#d9534f", "#2f8f78", "#f2a93b", "#7a4b94", "#8a8f98", "#17becf", "#bcbd22"]
fig = go.Figure()
for i in range(K_MODES):
    s_i = smp[smp["mode"] == i]
    fig.add_trace(go.Scattergl(x=s_i["x"], y=s_i["y"], mode="markers", name=f"{LAB[i]} (n={int((Mx['mode'] == i).sum()):,})", marker=dict(size=4, opacity=0.55, color=PALM[i % len(PALM)]),
                               customdata=np.c_[s_i["maia_p_top1"], s_i["maia_p_true"], s_i["maia_rank_true"], s_i["elo_self"], s_i["time_spent"]],
                               hovertemplate="Maia conf %{customdata[0]:.0%}<br>P(human move) %{customdata[1]:.2%} (rank %{customdata[2]:.0f})<br>Elo %{customdata[3]:.0f} · thought %{customdata[4]:.0f}s<extra></extra>"))
fig.update_layout(height=660, title=f"Map of Maia's misses ({EMB} of the behavioural signature, {len(smp):,} sampled misses) – click a cluster", legend=dict(orientation="h", y=-0.12),
                  xaxis=dict(showticklabels=False, title=f"{EMB}-1"), yaxis=dict(showticklabels=False, title=f"{EMB}-2"))
register(fig, "Failure modes", "Failure-mode map (static)", "", show=True)

fig = go.Figure(go.Heatmap(z=zprof.values, x=[WORDS[f][0] + " →" for f in zprof.columns], y=[LAB[i] for i in zprof.index], colorscale="RdBu_r", zmid=0, text=np.round(zprof.values, 2), texttemplate="%{text}",
                           colorbar=dict(title="z")))
fig.update_layout(height=120 + 70 * K_MODES, title="Signature profile of each failure mode (z-score vs. all misses; red = higher than typical)", margin=dict(l=300))
register(fig, "Failure modes", "Mode profile", "Read each row as a description of a kind of miss: e.g. high 'Maia confident' and low 'human move plausible' = confidently wrong; the opposite = a toss-up Maia lost.")

# density / spread: why is a mode dense or sparse?
dens = pd.DataFrame({"mode": [LAB[i] for i in range(K_MODES)], "share_of_misses": [(Mx["mode"] == i).mean() for i in range(K_MODES)],
                     "rel_density": [med_knn / max(np.median(smp.loc[smp["mode"] == i, "knn"]), 1e-9) for i in range(K_MODES)]})
fig = make_subplots(1, 2, subplot_titles=["share of all misses", "relative density in signature space (1 = typical)"], shared_yaxes=True)
fig.add_trace(go.Bar(x=dens["share_of_misses"], y=dens["mode"], orientation="h", marker_color=C_MAIA), 1, 1)
fig.add_trace(go.Bar(x=dens["rel_density"], y=dens["mode"], orientation="h", marker_color=C_HUMAN), 1, 2); fig.add_vline(x=1, line_dash="dash", row=1, col=2)
fig.update_layout(height=120 + 60 * K_MODES, showlegend=False, margin=dict(l=300), title="Big ≠ dense: how common a failure mode is vs. how tightly its members resemble each other")
register(fig, "Failure modes", "Mode size vs density", "A mode that is large but sparse is a catch-all; a small but dense mode is a precise, repeatable mechanism worth Phase-II attention.")

# hierarchy: modes -> sub-clusters (colour = human blunder rate)
ids, labs, pars, vals_, cols_ = [], [], [], [], []
for i in range(K_MODES):
    rows_i = Mx[Mx["mode"] == i]; qd = rows_i.dropna(subset=["move_quality"])
    ids.append(f"m{i}"); labs.append(LAB[i]); pars.append(""); vals_.append(len(rows_i)); cols_.append((qd["move_quality"] == "blunder").mean())
    for sc in sorted(rows_i["sub"].unique()):
        r_ = rows_i[rows_i["sub"] == sc]; qs = r_.dropna(subset=["move_quality"])
        ids.append(f"m{i}.{sc}"); labs.append(f"sub {sc}"); pars.append(f"m{i}"); vals_.append(len(r_)); cols_.append((qs["move_quality"] == "blunder").mean())
fig = go.Figure(go.Sunburst(ids=ids, labels=labs, parents=pars, values=vals_, branchvalues="total", marker=dict(colors=cols_, colorscale="YlOrRd", colorbar=dict(title="human<br>blunder rate")),
                            hovertemplate="%{label}<br>n=%{value:,}<br>blunder rate %{color:.1%}<extra></extra>"))
fig.update_layout(height=640, title="Failure modes and their sub-clusters (colour = how often the human's move was an engine blunder)")
register(fig, "Failure modes", "Mode hierarchy", "Sub-clusters of one mode with very different colours are the same kind of miss with different causes (e.g. human error vs. Maia error).")

# context of each mode
ctx = make_subplots(1, 2, subplot_titles=["phase", "player Elo category"], shared_yaxes=True)
for c, col in (("phase", 1), ("maia_elo_cat_self", 2)):
    order = PHASES if c == "phase" else ELO_ORDER
    ct = pd.crosstab(Mx["mode_label"], Mx[c].astype(str), normalize="index").reindex(index=[LAB[i] for i in range(K_MODES)], columns=order).fillna(0)
    for j, v in enumerate(order):
        ctx.add_trace(go.Bar(y=ct.index, x=ct[v], orientation="h", name=v, legendgroup=f"{c}{v}", showlegend=True), 1, col)
ctx.update_layout(barmode="stack", height=130 + 70 * K_MODES, margin=dict(l=300), title="Where does each failure mode happen?")
register(ctx, "Failure modes", "Mode context", "Modes that live in one phase or one Elo band point to a condition-specific mechanism; modes spread evenly are condition-independent.")

## 7.3 Input-space map – *where* does Maia fail, and how is that linked to *how* it fails?
§7.2 clusters only the **misses**, in Maia's **output** space (confidence, entropy, rank of the human move). That has three limits: a cluster "distinguished by confidence" is so by construction; with no correct predictions in the picture it cannot show *where* the model fails more; and "dense/sparse" there describes how alike the misses are, not how common the *situations* are. This section fixes this:

1. **Input space.** Every sampled position (hits **and** misses) is described only by what Maia receives: the board (12 piece planes, mirrored to the side to move, as Maia does) and the two Elo labels, plus legal-move count and material balance. Clock and think-time are *not* part of this space. A kNN graph, a 2-D embedding and K-means **regions** are built on it.
2. **Density vs. failure.** Exact repeats (opening-book positions) and kNN sparsity are tested against the failure rate (with game-clustered CIs; sparsity also stratified by phase). A game-grouped AUC says how much failure is predictable from the input alone.
3. **Where ↔ how.** Every §7.2 failure mode is placed on the input map and described by *input* features measured against **all** positions (not circular). Region × mode shows, per region, how many positions per 100 end in a miss of mode *m*; Cramér's V says how strongly region and mode are associated.
4. **Dashboard.** New tab *Input space*: click a region to see its dossier (what defines it, how Maia fails there, dense vs. sparse half) and board examples.

*Caveats:* the embedding axes are not interpretable (only neighbourhoods are); kNN density is relative to the 80k-position sample; sparsity is partly confounded with phase and Elo, which is why the stratified view is shown; chi-square p-values ignore game clustering (use V as the effect size).

In [33]:
# ===== 7.3 (a) Input space: what Maia sees (board + Elo) for ALL positions, kNN density, embedding, regions =====
from sklearn.decomposition import TruncatedSVD
from scipy import sparse
from scipy.stats import chi2_contingency

S_IN, N_SVD, K_NN, N_REG = 80_000, 24, 50, 12
W_BOARD, W_SIDE = 1.0, 0.5        # relative weight of the board block vs the (Elo, legal moves, material) block in the distance
PIECES = "PNBRQKpnbrqk"; PIDX = {c: i for i, c in enumerate(PIECES)}

def fen_planes(fen):
    # indices of the 1-entries of the 12x64 piece-plane vector, from the side-to-move's perspective (Maia mirrors the board for Black)
    parts = fen.split(" "); rows = parts[0].split("/")
    if parts[1] == "b": rows = [r.swapcase() for r in rows[::-1]]
    out, sq = [], 0
    for r in rows:
        for ch in r:
            if ch.isdigit(): sq += int(ch)
            else: out.append(PIDX[ch] * 64 + sq); sq += 1
    return out

IS = d.sample(min(S_IN, len(d)), random_state=SEED).copy()          # hits AND misses
IS["pop"] = IS["key"].map(d["key"].value_counts()); IS["pop_log"] = np.log10(IS["pop"])   # how often this exact position occurs in the whole data
idx_cols, ptr = [], [0]
for f in tqdm(IS["fen"], desc="board planes"):
    idx_cols += fen_planes(f); ptr.append(len(idx_cols))
Pm = sparse.csr_matrix((np.ones(len(idx_cols)), idx_cols, ptr), shape=(len(IS), 12 * 64))
Zb = TruncatedSVD(N_SVD, random_state=SEED).fit_transform(Pm)
Zb = Zb / np.sqrt((Zb ** 2).sum(1).mean()) * W_BOARD
SIDE = IS[["elo_self", "elo_oppo", "n_legal", "mat_balance"]].astype(float).assign(n_legal=lambda x: np.log(x["n_legal"]))
Zs = StandardScaler().fit_transform(SIDE) * (W_SIDE / np.sqrt(SIDE.shape[1]))
Zin = np.hstack([Zb, Zs])                                            # input space: only things Maia is given (+ legal moves / material, derived from the board)

yI = IS["failed"].to_numpy(float); gcode_IS = pd.factorize(IS["game_id"])[0]
nn_in = NearestNeighbors(n_neighbors=K_NN + 1, n_jobs=-1).fit(Zin)
full_d, full_n = nn_in.kneighbors(Zin)
self_m = full_n == np.arange(len(Zin))[:, None]                      # drop each point itself (duplicates can tie with distance 0)
self_m[~self_m.any(1), -1] = True
keep = ~self_m
nbr = full_n[keep].reshape(len(Zin), K_NN); dist_in = full_d[keep].reshape(len(Zin), K_NN)
IS["knn_dist"] = dist_in.mean(1)                                     # sparsity: large = rare region of position space
IS["local_fail"] = yI[nbr].mean(1)                                   # failure rate among the 50 most similar positions

try:
    import umap; emb = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=SEED).fit_transform(Zin); EMB_IN = "UMAP"
except Exception:
    emb = PCA(2, random_state=SEED).fit_transform(Zin); EMB_IN = "PCA"
IS["ex"], IS["ey"] = emb[:, 0], emb[:, 1]

# --- regions of position space (K-means on the input space) and interpretable descriptions ---
INF = ["n_pieces", "mat_balance", "n_legal", "elo_self", "elo_oppo", "elo_diff", "queens_on", "in_check", "move_number", "pop_log"]
INF_NAMES = dict(n_pieces="pieces on board", mat_balance="material balance", n_legal="legal moves", elo_self="player Elo", elo_oppo="opponent Elo", elo_diff="Elo − opp. Elo",
                 queens_on="queens on", in_check="in check", move_number="move no.", pop_log="position popularity")
WORDS_IN = dict(n_pieces=("many pieces", "few pieces"), mat_balance=("ahead in material", "behind in material"), n_legal=("many legal moves", "few legal moves"),
                elo_self=("high Elo", "low Elo"), elo_oppo=("strong opponent", "weak opponent"), elo_diff=("higher-rated than opp.", "lower-rated than opp."),
                queens_on=("queens on", "queens off"), in_check=("in check", "not in check"), move_number=("late", "early"), pop_log=("book position", "rare position"))
XI = IS[INF].astype(float).to_numpy(); sdI = XI.std(0) + 1e-9
def smd_vs_rest(mask): return (XI[mask].mean(0) - XI[~mask].mean(0)) / sdI
def label_from_smd(smd, k=2):
    top = np.argsort(-np.abs(smd))[:k]
    return " & ".join(WORDS_IN[INF[j]][0 if smd[j] > 0 else 1] for j in top)

km_in = KMeans(N_REG, n_init=5, random_state=SEED).fit(Zin)
order_in = pd.Series(km_in.labels_).value_counts().index.tolist()
IS["region"] = pd.Series(km_in.labels_).map({c: i for i, c in enumerate(order_in)}).values      # 0 = largest region
REG_LAB = {r: f"R{r}: {label_from_smd(smd_vs_rest(IS['region'].values == r))}" for r in range(N_REG)}
IS["region_label"] = IS["region"].map(REG_LAB)

# --- failure mode (from 7.2) of every sampled MISS; NaN for hits ---
IS["mode"] = Mx.set_index("orig_idx")["mode"].reindex(IS.index)
print(f"{len(IS):,} sampled positions | {int(IS['mode'].notna().sum()):,} misses carry a failure-mode label | embedding: {EMB_IN}")

# --- how predictable is failure from the INPUT alone (board + Elo), game-grouped CV? ---
Xin = np.c_[Zin, IS[["n_legal", "mat_balance", "n_pieces"]].to_numpy(float)]
Xin_r = np.c_[Xin, IS["knn_dist"].to_numpy(), IS["pop_log"].to_numpy()]
def auc_in(Xm):
    out = []
    for tr, te in GroupKFold(5).split(Xm, groups=IS["game_id"]):
        m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=SEED).fit(Xm[tr], yI[tr])
        out.append(roc_auc_score(yI[te], m.predict_proba(Xm[te])[:, 1]))
    return np.array(out)
a_in, a_in_r = auc_in(Xin), auc_in(Xin_r)
print(f"AUC for 'Maia misses', input space only: {a_in.mean():.3f} ± {a_in.std():.3f} | + rarity features (kNN sparsity, popularity): {a_in_r.mean():.3f} ± {a_in_r.std():.3f}")

board planes:   0%|          | 0/80000 [00:00<?, ?it/s]

80,000 sampled positions | 40,350 misses carry a failure-mode label | embedding: PCA
AUC for 'Maia misses', input space only: 0.621 ± 0.002 | + rarity features (kNN sparsity, popularity): 0.620 ± 0.002


In [34]:
# ===== 7.3 (b) Maps and the density-vs-failure test =====
import plotly.colors as pc
PALR = (pc.qualitative.Dark24 + pc.qualitative.Light24)[:N_REG]
G_IN, MIN_CELL = 40, 25

def grid_stat(x, yv, v, G=G_IN, min_n=MIN_CELL):
    xe, ye = np.linspace(x.min(), x.max(), G + 1), np.linspace(yv.min(), yv.max(), G + 1)
    ix = np.clip(np.digitize(x, xe) - 1, 0, G - 1); iy = np.clip(np.digitize(yv, ye) - 1, 0, G - 1)
    n, s = np.zeros((G, G)), np.zeros((G, G))
    np.add.at(n, (iy, ix), 1); np.add.at(s, (iy, ix), v)
    with np.errstate(invalid="ignore", divide="ignore"): m = np.where(n >= min_n, s / n, np.nan)
    return (xe[:-1] + xe[1:]) / 2, (ye[:-1] + ye[1:]) / 2, n, m

# (1) the input-space map: where is the failure rate high, where is the data, where are the book positions?
gx, gy, n_g, rate_g = grid_stat(IS["ex"].values, IS["ey"].values, yI)
_, _, _, pop_g = grid_stat(IS["ex"].values, IS["ey"].values, IS["pop_log"].values)
fig = make_subplots(1, 3, subplot_titles=["failure rate (hits AND misses)", "how many positions (log10)", "how often the exact position repeats (log10)"], horizontal_spacing=0.08)
fig.add_trace(go.Heatmap(z=rate_g, x=gx, y=gy, colorscale="RdBu_r", zmid=GLOBAL_FAIL, zmin=GLOBAL_FAIL - 0.2, zmax=GLOBAL_FAIL + 0.2, customdata=n_g,
                         colorbar=dict(x=0.29, len=0.8, title="failure"), hovertemplate="failure %{z:.3f}<br>n=%{customdata:.0f}<extra></extra>"), 1, 1)
fig.add_trace(go.Heatmap(z=np.where(n_g >= MIN_CELL, np.log10(np.maximum(n_g, 1)), np.nan), x=gx, y=gy, colorscale="Viridis", colorbar=dict(x=0.645, len=0.8, title="log10 n")), 1, 2)
fig.add_trace(go.Heatmap(z=pop_g, x=gx, y=gy, colorscale="Magma", colorbar=dict(x=1.0, len=0.8, title="log10 repeats")), 1, 3)
fig.update_xaxes(showticklabels=False); fig.update_yaxes(showticklabels=False)
fig.update_layout(height=470, title=f"Input-space map ({EMB_IN} of board + Elo; {len(IS):,} positions; cells with n<{MIN_CELL} hidden)")
register(fig, "Input space", "Input-space map", "Left: the failure rate of ALL positions in each neighbourhood. Middle/right: how crowded the region is, and whether it is made of repeated (book) positions. Axes are not interpretable, neighbourhoods are.")

# (2) does the failure rate depend on how common / rare the situation is?
IS["pop_bin"] = pd.cut(IS["pop"], [0, 1, 4, 19, 99, np.inf], labels=["seen once", "2–4×", "5–19×", "20–99×", "≥100×"])
POPL = [str(c) for c in IS["pop_bin"].cat.categories]
nb = IS[IS["pop"] <= 4].copy()                                       # non-book positions: sparsity is not just 'repeated opening'
nb["sparse_q"] = pd.qcut(nb["knn_dist"].rank(method="first"), 5, labels=["Q1 dense", "Q2", "Q3", "Q4", "Q5 sparse"])
SQL = [str(c) for c in nb["sparse_q"].cat.categories]
fig = make_subplots(1, 3, subplot_titles=["exact repeats of the position", "kNN sparsity (non-book positions)", "sparsity × phase (non-book)"], horizontal_spacing=0.09, specs=[[{}, {}, {}]])
for ci, (df_, col, order) in enumerate([(IS, "pop_bin", POPL), (nb, "sparse_q", SQL)], start=1):
    r = cluster_rates(df_, col).reindex(order)
    fig.add_trace(go.Bar(x=order, y=r["rate"], marker_color=C_MAIA, showlegend=False, customdata=r["n"], hovertemplate="%{x}<br>failure %{y:.3f}<br>n=%{customdata:,}<extra></extra>",
                         error_y=dict(type="data", symmetric=False, array=r["hi"] - r["rate"], arrayminus=r["rate"] - r["lo"])), 1, ci)
    fig.add_hline(y=GLOBAL_FAIL, line_dash="dash", line_color="black", row=1, col=ci)
ht = heat_trace(cluster_rates(nb, ["phase", "sparse_q"]), "phase", "sparse_q", PHASES, SQL, min_n=100, zmin=GLOBAL_FAIL - 0.15, zmax=GLOBAL_FAIL + 0.15, zmid=GLOBAL_FAIL)
ht.showscale = False; fig.add_trace(ht, 1, 3)
fig.update_layout(height=450, yaxis_title="failure rate", title="Does Maia fail more in rare regions of position space? (95% CI, bootstrap over games)")
register(fig, "Input space", "Rarity vs failure",
         "Left: opening-book positions (seen many times) vs. unique ones. Middle: among non-book positions, failure across kNN-sparsity quintiles. Right: the same inside each phase (guards against 'sparse = endgame').")
rar = cluster_rates(nb, "sparse_q").reindex(SQL)
print(f"non-book positions: failure Q1 dense {rar.loc[SQL[0], 'rate']:.3f} [{rar.loc[SQL[0], 'lo']:.3f}, {rar.loc[SQL[0], 'hi']:.3f}]  vs  Q5 sparse {rar.loc[SQL[-1], 'rate']:.3f} [{rar.loc[SQL[-1], 'lo']:.3f}, {rar.loc[SQL[-1], 'hi']:.3f}]")

# (3) the failure modes of 7.2 placed on the input map (hits in grey for context)
hits = IS[IS["failed"] == False]; hits = hits.sample(min(15000, len(hits)), random_state=SEED)
fig = go.Figure(go.Scattergl(x=hits["ex"], y=hits["ey"], mode="markers", marker=dict(size=3, color="#d5d9e0", opacity=0.5), name="Maia correct", hoverinfo="skip"))
for i in range(K_MODES):
    s_i = IS[IS["mode"] == i]
    fig.add_trace(go.Scattergl(x=s_i["ex"], y=s_i["ey"], mode="markers", name=f"{LAB[i]} (n={len(s_i):,})", marker=dict(size=3.5, opacity=0.6, color=PALM[i % len(PALM)]),
                               customdata=np.c_[s_i["maia_p_top1"], s_i["maia_p_true"], s_i["region_label"]],
                               hovertemplate="Maia conf %{customdata[0]:.0%} · P(human move) %{customdata[1]:.2%}<br>%{customdata[2]}<extra></extra>"))
fig.update_layout(height=680, title="Where do the failure modes live in INPUT space? (grey = correct predictions)", legend=dict(orientation="h", y=-0.1),
                  xaxis=dict(showticklabels=False, title=f"{EMB_IN}-1 of board + Elo"), yaxis=dict(showticklabels=False, title=f"{EMB_IN}-2"))
register(fig, "Input space", "Failure modes on the input map",
         "If a mode forms its own island, 'how' Maia fails is tied to 'where'; if modes are scattered evenly over the map they are condition-independent. Click legend entries to isolate a mode.")

# (4) clickable region map (the dashboard links it to the region dossier)
fig = go.Figure()
for r in range(N_REG):
    s_r = IS[IS["region"] == r]; s_r = s_r.sample(min(len(s_r), 6000), random_state=SEED)
    fig.add_trace(go.Scattergl(x=s_r["ex"], y=s_r["ey"], mode="markers", name=REG_LAB[r], marker=dict(size=3.5, opacity=0.6, color=PALR[r]),
                               customdata=np.c_[s_r["failed"].astype(int), s_r["maia_p_top1"]], hovertemplate=REG_LAB[r] + "<br>Maia conf %{customdata[1]:.0%} · miss=%{customdata[0]}<extra></extra>"))
fig.update_layout(height=680, title=f"Regions of position space ({EMB_IN}; K-means on board + Elo) – click a region", legend=dict(orientation="h", y=-0.12),
                  xaxis=dict(showticklabels=False, title=f"{EMB_IN}-1"), yaxis=dict(showticklabels=False, title=f"{EMB_IN}-2"))
register(fig, "Input space", "Input-space region map (static)", "", show=True)

non-book positions: failure Q1 dense 0.467 [0.459, 0.475]  vs  Q5 sparse 0.477 [0.469, 0.485]


In [35]:
# ===== 7.3 (c) Regions: how much do they fail, how are they linked to the failure modes, dossiers =====
reg_r = cluster_rates(IS, "region").reindex([str(r) for r in range(N_REG)])
reg_gap = IS.groupby("region").apply(lambda s: s["maia_p_top1"].mean() - s["maia_correct"].mean()).reindex(range(N_REG))
med_knn_in = np.median(IS["knn_dist"])
reg_dens = pd.Series({r: med_knn_in / max(np.median(IS.loc[IS["region"] == r, "knn_dist"]), 1e-9) for r in range(N_REG)})
reg_pop = IS.groupby("region")["pop_log"].mean().reindex(range(N_REG))

reg_tab = pd.DataFrame({"region": [REG_LAB[r] for r in range(N_REG)], "n": reg_r["n"].values, "failure": reg_r["rate"].values, "lo": reg_r["lo"].values, "hi": reg_r["hi"].values,
                        "lift": reg_r["rate"].values / GLOBAL_FAIL, "rel_density": reg_dens.values, "overconf_gap": reg_gap.values, "mean_log10_repeats": reg_pop.values})
display(reg_tab.round(3))

labs = [f"{REG_LAB[r]} (n={int(reg_r.loc[str(r), 'n']):,})" for r in range(N_REG)]
fig = make_subplots(1, 4, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=["failure rate (95% CI)", "relative density (1 = typical)", "over-confidence gap", "mean log10 repeats"])
fig.add_trace(go.Bar(y=labs, x=reg_r["rate"], orientation="h", marker_color=C_MAIA, error_x=dict(type="data", symmetric=False, array=reg_r["hi"] - reg_r["rate"], arrayminus=reg_r["rate"] - reg_r["lo"])), 1, 1)
fig.add_vline(x=GLOBAL_FAIL, line_dash="dash", row=1, col=1)
fig.add_trace(go.Bar(y=labs, x=reg_dens.values, orientation="h", marker_color=C_HUMAN), 1, 2); fig.add_vline(x=1, line_dash="dash", row=1, col=2)
fig.add_trace(go.Bar(y=labs, x=reg_gap.values, orientation="h", marker_color=C_GREY), 1, 3)
fig.add_trace(go.Bar(y=labs, x=reg_pop.values, orientation="h", marker_color="#7a4b94"), 1, 4)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=140 + 55 * N_REG, showlegend=False, margin=dict(l=380), title="Regions of position space: how much is Maia wrong there, and why does each region hold this much data?")
register(fig, "Input space", "Region summary",
         "Failure is computed over ALL positions of a region. Compare failure with relative density and with repeats: are the rare or the book-heavy regions the failing ones?")

# --- link 'where' ↔ 'how': positions per 100 that are a miss of mode m, by region ---
M_ct = pd.crosstab(IS["region"], IS["mode"]).reindex(index=range(N_REG), columns=range(K_MODES)).fillna(0)
n_reg = IS["region"].value_counts().reindex(range(N_REG)).values
rate_rm = M_ct.div(n_reg, axis=0)                                    # P(miss of mode m | region), denominator = ALL positions in the region
glob_m = M_ct.sum(0) / len(IS); lift_rm = rate_rm / glob_m
chi2_, p_, dof_, _ = chi2_contingency(M_ct.values + 1e-9)
V = float(np.sqrt(chi2_ / (M_ct.values.sum() * (min(M_ct.shape) - 1))))
print(f"region × failure-mode association among misses: Cramér's V = {V:.3f} (≈0.05 weak, ≈0.15 moderate, ≥0.25 strong; chi-square p ignores game clustering)")
Zl = np.where(M_ct.values >= 30, lift_rm.values, np.nan)
Tl = np.where(M_ct.values >= 30, np.char.mod("%.1f", rate_rm.values * 100), "")
fig = go.Figure(go.Heatmap(z=Zl, x=[LAB[i] for i in range(K_MODES)], y=labs, text=Tl, texttemplate="%{text}", colorscale="RdBu_r", zmid=1, zmin=0, zmax=3,
                           colorbar=dict(title="lift vs. overall"), hovertemplate="%{y}<br>%{x}<br>lift ×%{z:.2f}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(tickangle=-30)
fig.update_layout(height=170 + 52 * N_REG, margin=dict(l=380, b=190), title=f"Where × how: misses of each mode per 100 positions of the region (cell text), colour = lift over the overall rate (Cramér's V = {V:.2f})")
register(fig, "Input space", "Region × failure mode",
         "Cells with fewer than 30 misses are blank. A red cell means this kind of miss is concentrated in this part of position space; a row that is uniformly white means the region fails in the same mixture of ways as everywhere.")

# --- what distinguishes each failure mode in INPUT space, measured against ALL positions (not against other misses) ---
PM = np.vstack([(XI[IS["mode"].values == i].mean(0) - XI.mean(0)) / sdI for i in range(K_MODES)])
fig = go.Figure(go.Heatmap(z=PM, x=[INF_NAMES[f] for f in INF], y=[LAB[i] for i in range(K_MODES)], colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1, text=np.round(PM, 2), texttemplate="%{text}",
                           colorbar=dict(title="sd vs. all positions")))
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=150 + 62 * K_MODES, margin=dict(l=330), title="Failure modes described by INPUT features (standardised difference of the mode's misses to ALL positions)")
register(fig, "Input space", "Failure modes: input profile",
         "Non-circular description of a mode: it uses only conditions (position, Elo, popularity), none of Maia's own outputs. Rows with large entries are modes tied to a particular kind of situation.")

# --- region dossiers (clickable in the dashboard) ---
def region_dossier(r):
    m = IS["region"].values == r; idx = np.where(m)[0]; sub = IS.iloc[idx]; n = len(idx); rt = reg_r.loc[str(r)]
    smd = smd_vs_rest(m); top = np.argsort(-np.abs(smd))[:6]
    bars = [(INF_NAMES[INF[j]], smd[j], f"{smd[j]:+.2f} sd") for j in top]
    mm = M_ct.loc[r]; share = (mm / max(mm.sum(), 1)).sort_values(ascending=False)
    mode_lines = "".join(f"<div class='sub'>• {LAB[int(i)]}: {s:.0%} of the misses here (×{lift_rm.loc[r, i]:.1f} vs. everywhere) · {rate_rm.loc[r, i] * 100:.1f} per 100 positions</div>" for i, s in share.head(3).items())
    knn_v = sub["knn_dist"].to_numpy(); med = np.median(knn_v); halves = []
    for nm, ii in (("denser half", idx[knn_v <= med]), ("sparser half", idx[knn_v > med])):
        lo, hi = ci_idx(gcode_IS, yI, ii); halves.append((nm, ii, yI[ii].mean(), lo, hi))
    differ = halves[0][3] > halves[1][4] or halves[1][3] > halves[0][4]
    half_txt = "".join(f"<div class='sub'>• {nm}: n={len(ii):,}, failure {rt_:.1%} [{lo:.1%}, {hi:.1%}]</div>" for nm, ii, rt_, lo, hi in halves)
    cm = [IS["mode"].iloc[ii].value_counts().reindex(range(K_MODES)).fillna(0).to_numpy() for _, ii, *_ in halves]
    jtxt = ""
    if min(c.sum() for c in cm) >= 30: jtxt = f" · mode mix differs between halves (JSD {jsd_np(cm[0] / cm[0].sum(), cm[1] / cm[1].sum()):.3f}; &gt;0.02 = visibly)"
    verdict = "failure differs between dense and sparse half (CIs do not overlap)" if differ else "failure rate not distinguishable between dense and sparse half"
    html = [f"<div class='dos'><b>{REG_LAB[r]}</b><div class='sub'>{n:,} positions = {n / len(IS):.1%} of the sample · failure {rt['rate']:.1%} [{rt['lo']:.1%}, {rt['hi']:.1%}] · lift ×{rt['rate'] / GLOBAL_FAIL:.2f} · "
            f"over-confidence gap {reg_gap[r]:+.3f}</div>",
            "<h4>What defines this region (input features vs. all other regions)</h4>", bars_html(bars, 1.5),
            "<h4>How does Maia fail here?</h4>", mode_lines or "<div class='sub'>too few misses</div>",
            "<h4>Why this much data? Dense or sparse?</h4>",
            f"<div class='sub'>relative density ×{reg_dens[r]:.2f} (1 = typical, &gt;1 = positions here are unusually similar to each other) · mean log10 repeats {reg_pop[r]:.2f} "
            f"({(sub['pop'] >= 5).mean():.0%} of positions occur ≥5 times)</div>",
            "<h4>Does density matter inside the region?</h4>", half_txt, f"<div class='sub'>{verdict}{jtxt}</div></div>"]
    mi = sub[sub["failed"]]
    conf = mi.nlargest(2, "maia_p_top1"); rest = mi.drop(conf.index); rest = rest.sample(min(3, len(rest)), random_state=SEED)
    cards = [card_html(row, f"confident miss · M{int(row['mode'])}") for _, row in conf.iterrows()] + [card_html(row, f"random miss · M{int(row['mode'])}") for _, row in rest.iterrows()]
    return dict(html="".join(html), cards=cards)

REGIONS = {str(r): region_dossier(r) for r in tqdm(range(N_REG), desc="region dossiers")}
print(f"{N_REG} region dossiers ready – click a region in the dashboard's 'Input space' tab")

,region,n,failure,lo,hi,lift,rel_density,overconf_gap,mean_log10_repeats
0,R0: many legal moves & rare position,10870,0.496,0.487,0.505,0.987,0.966,0.039,0.001
1,R1: low Elo & weak opponent,8410,0.599,0.590,0.610,1.193,1.318,0.047,0.942
2,R2: many pieces & early,8200,0.471,0.459,0.481,0.937,0.967,0.022,0.062
3,R3: low Elo & weak opponent,8077,0.530,0.519,0.540,1.054,0.891,0.036,0.002
4,R4: late & few pieces,7850,0.476,0.464,0.488,0.947,1.159,0.041,0.005
5,R5: book position & early,7080,0.630,0.620,0.640,1.254,1.721,0.111,1.809
6,R6: many pieces & early,6744,0.505,0.494,0.517,1.005,0.930,0.026,0.012
7,R7: strong opponent & high Elo,6534,0.496,0.485,0.509,0.987,0.899,0.042,0.006
8,R8: few pieces & late,6522,0.482,0.469,0.494,0.959,1.125,0.039,0.004
9,R9: in check & few legal moves,4586,0.303,0.290,0.316,0.603,0.947,0.036,0.010


region × failure-mode association among misses: Cramér's V = 0.371 (≈0.05 weak, ≈0.15 moderate, ≥0.25 strong; chi-square p ignores game clustering)


region dossiers:   0%|          | 0/12 [00:00<?, ?it/s]

12 region dossiers ready – click a region in the dashboard's 'Input space' tab


## 8. Dashboard – one linked, interactive HTML file
Collects every registered figure into tabs, links the **atlas** to real board examples (click a block), and appends the **AI log** so the documentation travels with the visualization. Open `ai_phase1_out/maia2_failure_atlas.html` in a browser (self-contained, works offline).

In [36]:
from plotly.offline import get_plotlyjs

TAB_ORDER = ["Failure atlas", "Failure modes", "Input space", "Anatomy of a failure", "Noise ceiling", "Clock & history", "Elo response", "Move style", "Audit", "AI log"]
HTML_T = """<!doctype html><html><head><meta charset="utf-8"><title>Maia-2 failure atlas</title>
<style>
body{font-family:-apple-system,Segoe UI,Helvetica,Arial,sans-serif;margin:0;color:#1d2330;background:#f6f7f9}
header{padding:14px 22px;background:#1d2330;color:#fff}header h1{margin:0;font-size:20px}header p{margin:4px 0 0;font-size:13px;opacity:.8}
#tabs{display:flex;flex-wrap:wrap;gap:4px;padding:8px 18px;background:#e9ecf1;position:sticky;top:0;z-index:5}
#tabs button{border:0;padding:8px 14px;border-radius:6px;background:#fff;cursor:pointer;font-size:14px}#tabs button.on{background:#2a78d6;color:#fff}
.panel{display:none;padding:14px 22px 40px}.fig{background:#fff;border-radius:8px;padding:10px 14px;margin:0 0 18px;box-shadow:0 1px 3px rgba(0,0,0,.08)}
.fig h3{margin:4px 0}.fig .cap{font-size:13px;color:#566}
.atlasrow{display:flex;gap:16px;align-items:flex-start}.atlasrow .left{flex:1.1;min-width:480px}.atlasrow .right{flex:1;min-width:420px}
#nodeinfo{background:#fff;border-radius:8px;padding:12px 14px;box-shadow:0 1px 3px rgba(0,0,0,.08);margin-bottom:12px;font-size:14px}
#examples{display:flex;flex-wrap:wrap;gap:10px}.card{background:#fff;border-radius:8px;padding:6px;width:240px;box-shadow:0 1px 3px rgba(0,0,0,.08)}
.card .cap{font-size:11.5px;padding:4px 2px;color:#334}.brow{display:flex;align-items:center;gap:6px;font-size:12px;margin:1px 0}.brow .bl{width:150px}.brow .bt{flex:1;background:#eef0f4;height:8px;border-radius:4px;overflow:hidden;display:block}.brow .bt i{display:block;height:8px}.brow .bv{width:70px;text-align:right;color:#445}.dos h4{margin:12px 0 4px;font-size:13px;border-top:1px solid #e3e6eb;padding-top:8px}.dos .sub{font-size:12px;color:#556;margin:2px 0 6px}table.log{border-collapse:collapse;font-size:12.5px;background:#fff}
table.log td,table.log th{border:1px solid #d6dae1;padding:5px 8px;vertical-align:top;max-width:340px}
</style><script>__PLOTLYJS__</script></head><body>
<header><h1>Maia-2: where and why does it miss the human move?</h1><p>Lichess 2024-06 · black-box analysis · AI-aided group. Click atlas blocks for board examples (blue arrow = Maia, red = human).</p></header>
<nav id="tabs">__TABBTNS__</nav>__PANELS__
<script>
const NODES = __NODES__;
const MODES = __MODES__;
const REGIONS = __REGIONS__;
function showRegion(id){const m=REGIONS[id]; if(!m) return; document.getElementById('reginfo').innerHTML=m.html; document.getElementById('regexamples').innerHTML=m.cards.join('')||'<i>no examples</i>';}
function showMode(id){const m=MODES[id]; if(!m) return; document.getElementById('modeinfo').innerHTML=m.html; document.getElementById('modeexamples').innerHTML=m.cards.join('')||'<i>no examples</i>';}
function openTab(i){
  document.querySelectorAll('.panel').forEach((p,j)=>p.style.display=(i===j)?'block':'none');
  document.querySelectorAll('#tabs button').forEach((b,j)=>b.classList.toggle('on',i===j));
  document.querySelectorAll('#panel'+i+' .js-plotly-plot').forEach(el=>Plotly.Plots.resize(el));
}
function pct(x){return (100*x).toFixed(1)+'%'}
function showNode(id){
  const n=NODES[id]; if(!n) return; const s=n.stats;
  document.getElementById('nodeinfo').innerHTML='<b>'+n.path+'</b><br>n = '+s.n.toLocaleString()+' · rate '+pct(s.rate)+' (95% CI '+pct(s.lo)+'–'+pct(s.hi)+') · lift ×'+s.lift.toFixed(2)+
    '<br>Maia top-3 miss '+pct(s.top3_miss)+' · mean Maia confidence '+pct(s.mean_conf)+' · mean normalised entropy '+s.mean_entropy.toFixed(2)+
    (s.human_blunder===null?'':' · human blunder rate '+pct(s.human_blunder))+' · median think-time '+s.median_think.toFixed(1)+' s'+(n.dossier||'');
  document.getElementById('examples').innerHTML=n.cards.join('')||'<i>no examples</i>';
}
window.addEventListener('load',()=>{
  const el=document.getElementById('atlas_fig');
  if(el&&el.on){el.on('plotly_click',ev=>{const p=ev.points&&ev.points[0]; if(p&&p.id!==undefined) showNode(String(p.id));}); showNode('0');}
  const em=document.getElementById('modes_fig');
  if(em&&em.on){em.on('plotly_click',ev=>{const p=ev.points&&ev.points[0]; if(p) showMode(String(p.curveNumber));}); showMode('0');}
  const er=document.getElementById('regions_fig');
  if(er&&er.on){er.on('plotly_click',ev=>{const p=ev.points&&ev.points[0]; if(p) showRegion(String(p.curveNumber));}); showRegion('0');}
  openTab(0);
});
</script></body></html>"""

def build_dashboard(path=OUT / "maia2_failure_atlas.html"):
    tabs = {}
    for item in sorted(REG.values(), key=lambda it: not it["title"].endswith("(static)")): tabs.setdefault(item["tab"], []).append(item)
    names = [t for t in TAB_ORDER if t in tabs or t == "AI log"] + [t for t in tabs if t not in TAB_ORDER]
    btns, panels = [], []
    for i, nm in enumerate(names):
        btns.append(f"<button onclick='openTab({i})'>{nm}</button>"); body = ""
        for item in tabs.get(nm, []):
            if item["title"] == "Atlas (static)":
                fig_html = item["fig"].to_html(full_html=False, include_plotlyjs=False, div_id="atlas_fig")
                body += (f"<div class='fig'><h3>Failure atlas</h3><div class='cap'>Click a block to see its statistics and real board examples. Click the top bar to zoom out.</div>"
                         f"<div class='atlasrow'><div class='left'>{fig_html}</div><div class='right'><div id='nodeinfo'></div><div id='examples'></div></div></div></div>")
            elif item["title"] == "Failure-mode map (static)":
                fig_html = item["fig"].to_html(full_html=False, include_plotlyjs=False, div_id="modes_fig")
                body += (f"<div class='fig'><h3>Failure modes of Maia-2</h3><div class='cap'>Every dot is a miss, placed by its behavioural signature. Click a cluster to read its dossier and see typical boards.</div>"
                         f"<div class='atlasrow'><div class='left'>{fig_html}</div><div class='right'><div id='modeinfo'></div><div id='modeexamples'></div></div></div></div>")
            elif item["title"] == "Input-space region map (static)":
                fig_html = item["fig"].to_html(full_html=False, include_plotlyjs=False, div_id="regions_fig")
                body += (f"<div class='fig'><h3>Regions of position space</h3><div class='cap'>Each colour is a region of the input space (board + Elo). Click a region to see what defines it, how Maia fails there, whether density matters inside it, and example misses.</div>"
                         f"<div class='atlasrow'><div class='left'>{fig_html}</div><div class='right'><div id='reginfo'></div><div id='regexamples'></div></div></div></div>")
            else:
                body += f"<div class='fig'><h3>{item['title']}</h3><div class='cap'>{item['caption']}</div>{item['fig'].to_html(full_html=False, include_plotlyjs=False)}</div>"
        if nm == "AI log":
            body += "<div class='fig'><h3>Generative-AI usage log</h3>" + pd.read_csv(LOG_PATH).to_html(index=False, classes="log", escape=True) + "</div>"
        panels.append(f"<section class='panel' id='panel{i}'>{body}</section>")
    nodes_json = json.dumps(NODES).replace("</", "<\/"); regions_json = json.dumps(globals().get("REGIONS", {})).replace("</", "<\/"); modes_json = json.dumps(globals().get("MODES", {})).replace("</", "<\/")
    html = (HTML_T.replace("__PLOTLYJS__", get_plotlyjs()).replace("__TABBTNS__", "".join(btns))
                  .replace("__PANELS__", "".join(panels)).replace("__NODES__", nodes_json).replace("__MODES__", modes_json).replace("__REGIONS__", regions_json))
    path.write_text(html, encoding="utf-8"); print(f"wrote {path} ({path.stat().st_size/1e6:.1f} MB), tabs: {names}")

build_dashboard()

wrote ai_phase1_out/maia2_failure_atlas.html (21.0 MB), tabs: ['Failure atlas', 'Failure modes', 'Input space', 'Anatomy of a failure', 'Noise ceiling', 'Clock & history', 'Elo response', 'Move style', 'Audit', 'AI log']


## 9. Findings (fill in after running) – one line per hypothesis, always with the numbers and CIs
| | Verdict (supported / rejected / mixed) | Evidence (figure, number, CI) | What explains it? |
|---|---|---|---|
| H1 equal alternatives | | | |
| H2 noise ceiling | | | |
| H3 clock-blind | | | |
| H4 history-blind | | | |
| H5 Elo input | | | |
| H6 style bias | | | |
| H7 slices | | | |
| H8 artefacts | | | |
| H9 failure modes | | | |
| H10 input-space location | | | |

### Understanding protocol – fill one block per pattern you report (the dashboard pre-computes the evidence)
For each pattern (an atlas slice, a failure mode, a style bias, a think-time effect):
1. **What distinguishes it?** → atlas dossier *'what else differs'* / mode z-profile / contexts with lift. Write the explanation in words.
2. **Is it real after adjustment?** → *Adjusted vs raw effects*; held-out confirmation of sub-slices.
3. **Why is it dense or sparse?** → dossier *'why this much data'* / *Mode size vs density*. Is it an opening-book effect? A rare condition? A catch-all?
4. **Does it have sub-structure where Maia behaves differently?** → dossier *sub-slices* / *Mode hierarchy* (silhouette, JSD of predictions).
5. **Is it an error at all?** → engine and consensus checks (§2, §3).
6. **Where in input space does it live?** → *Input-space map*, *Rarity vs failure*, *Region × failure mode*, *Failure modes: input profile*. Is the failure tied to rare regions, to book positions, to a particular kind of position?
7. **Mechanism** (one sentence you could test in Phase II).

**Questions the task explicitly asks for each discovered pattern:** What distinguishes this slice from others? Why is it dense/sparse? Does it contain sub-slices where Maia behaves differently (use the atlas children)? Write the answers here, and log every AI-suggested interpretation you adopted in `ai_log.csv`.

**Hand-off to Phase II (white-box):** list the slices / positions where *inside* inspection would help (e.g. high-confidence Maia errors from §2, positions with high Elo-sensitivity from §5).